# Cosmic-Variance (CV) PSPEC view -- multiple sky realizations

Derived from `output_vis_sigloss_check_view.ipynb`. Instead of one `MODEL_DIR`, this reads a **list of realizations** (seeds) into lists of `combined_uvp_dict` / `combined_uvp_avg_dict`, then:

1. overlays the **0-delay PSPEC vs LST** for all seeds in one plot;
2. shows **individual Mode-Analytics** (LST-FFT + analytic centers) for `N_SAMPLE_REALIZATIONS` seeds; and
3. plots the **mean |FFT|** of the mode-analytic spectra averaged across realizations, with the analytic mode centers.

Imports, the model-dir read structure, the 0-delay PSPEC plot, and all LST-FFT / Mode-Analytics machinery are retained; bandstop / beam-FFT / percent-difference / derivation / appendix sections are dropped.

In [ ]:
# from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
import os, psutil 
def show_ram():
    proc = psutil.Process(os.getpid())
    rss = proc.memory_info().rss
    print(f"Notebook RAM usage: {rss/1e9:.2f} GB")

show_ram()


In [ ]:
import os
import gc
import glob
import numpy as np
import healpy as hp
import matplotlib
import hera_pspec as hp
from scipy import stats
import hera_cal as hc
from astropy import constants
from pyuvdata import utils as uvutils
from pyuvdata import UVData
import matplotlib.pyplot as plt
import itertools
from pyuvdata import UVData, UVCal

import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
from scipy import constants, interpolate
import copy
import glob
import re
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
pd.set_option('display.max_rows', 1000)
from uvtools.plot import plot_antpos, plot_antclass
from hera_qm import ant_metrics, ant_class, xrfi
from hera_cal import io, utils, redcal, apply_cal, datacontainer, abscal
from hera_filters import dspec
from IPython.display import display, HTML
import linsolve
# display(HTML("<style>.container { width:100% !important; }</style>"))
# _ = np.seterr(all='ignore')  # get rid of red warnings
# %config InlineBackend.figure_format = 'retina'

# this enables better memory management on linux
import ctypes
def malloc_trim():
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0) 
    except OSError:
        pass
    
import sys

from copy import deepcopy
from datetime import datetime
import time

matplotlib.rcParams["mathtext.fontset"] = "cm"
matplotlib.rcParams["font.family"] = "STIXGeneral"
matplotlib.rcParams["font.size"] = "18"
from matplotlib.ticker import MultipleLocator


####################################################################################################################################################################################################################################################################################

for repo in ['numpy', 'scipy', 'astropy', 'hera_cal', 'hera_qm', 'hera_filters', 'pyuvdata', 'hera_pspec']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')
    
# Old hera_filter version : ValueError: ridge_alpha is not a valid argument!valid arguments include ['suppression_factors', 'eigenval_cutoff', 'max_contiguous_edge_flags'] : Version hera_filters: 0.1.3
# Latest version working 24 Nov 24 : hera_filters: 0.1.6.dev1+g297dcce
# How to update :
# conda activate <hera>
# pip install --upgrade hera-filters
# python -c "import hera_filters; print(hera_filters.__version__)"

# pip install --upgrade hera-filters --user
# pip install git+https://github.com/HERA-Team/hera_filters.git

# pld hera_pspec: 0.4.2.dev4+gc34b82e

In [ ]:
# All H4C dependencies

# import numpy as np
import matplotlib.pyplot as plt
# To run pspecdata.pspec_run
from hera_pspec import PSpecContainer
from hera_pspec import utils as pspec_utils
from hera_pspec import pspecdata, pstokes
from hera_pspec import uvpspec

# To run io.HERAdata etc
from hera_cal import frf, delay_filter, io, smooth_cal

# import glob, tqdm, os, copy
import glob, os, copy
import functools
import time

from hera_qm import utils
from astropy import units

from matplotlib.colors import LogNorm
from hera_cal import apply_cal
from pyuvdata import UVBeam, UVData
from hera_pspec import grouping
import warnings
# To run datetime.now() function
from datetime import datetime
from hera_pspec.conversions import Cosmo_Conversions as cc
from hera_cal import lstbin
from hera_cal import frf
import scipy.interpolate as interp
from hera_cal.vis_clean import VisClean

from pathlib import Path

SDAY_KSEC = units.sday.to("ks")

In [ ]:
# import numpy as np
# np.set_printoptions(linewidth=370)  # or any large number that suits your screen


In [ ]:
pol_root=['xx']#, 'yy']
proc1 = ["cutbl"]#, "allbl"] "bwcut" "cutbl"
proc2 = ["cutlst"]#, "alllst"]

run_batch = ['250925']

chunk_min, chunk_max = 0, 288
fch_min, fch_max = 271, 276 

# sky_type = "ptsrc"
# sky_type = "eor"

In [ ]:
# ===========================================================================
# COSMIC-VARIANCE (CV) PARAMETERS  --  list of sky realizations (seeds)
# ===========================================================================
# Each entry is a MODEL_DIR (relative to BASE_OUTDIR) for ONE sky realization.
# They share the same instrument / beam / band; only the EoR sky seed differs,
# so averaging across them beats down cosmic variance.
#
# Auto-detected available CV ensemble for this band (fch0271-0276, D14m airy,
# fftvis_xcheck): seed700..seed707 each have saved Coh + Incoh PSPEC outputs.
import numpy as np

_CV_TEMPLATE = (
    "eor-grf-256/seed{seed}_freqslic_middle_fch0273ref/"                                    # fftvis_xcheck
    "nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt."
    "beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46."
    "_beammapperant_airyred-nonred_airyred/"
)

# Edit this list to choose which realizations to include.
from pathlib import Path
cand = [*range(700, 710), *range(900, 910)]
BASE_OUTDIR= Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/outputs")
REALIZATIONS = [_CV_TEMPLATE.format(seed=s) for s in cand
                if (BASE_OUTDIR / _CV_TEMPLATE.format(seed=s)).exists()]   # BASE_OUTDIR = your outputs root

# REALIZATIONS = [_CV_TEMPLATE.format(seed=s) for s in range(900, 910)]   # seed900 .. seed909

# Other available D14m-airy realizations (uncomment to add; data confirmed present):
# REALIZATIONS += ["eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/fftvis_xcheck/"
#                  "nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt."
#                  "beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46."
#                  "_beammapperant_airyred-nonred_airyred/"]
# REALIZATIONS += ["eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/"
#                  "nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt."
#                  "beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46."
#                  "_beammapperant_airyred-nonred_airyred/"]

# How many individual per-realization Mode-Analytics plots to draw (1, 4, ...).
N_SAMPLE_REALIZATIONS = 4

# Baseline group(s) to load (same convention as the source viewer notebook).
bl_len = [29.0]
bl_ang = [0.0]


def realization_label(model_dir):
    """Short seed label for legends, e.g. 'seed703' or 'rlzn_222'."""
    import re as _re
    s = str(model_dir)
    m = _re.search(r"seed(\d+)", s)
    if m:
        return f"seed{m.group(1)}"
    m = _re.search(r"rlzn_seed_(\d+)", s)
    return f"rlzn_{m.group(1)}" if m else s.split("/")[0]


def pick_sample_realizations(n_total, n_sample):
    """Evenly-spaced indices of the realizations to show individually."""
    n_sample = max(1, min(int(n_sample), int(n_total)))
    if n_sample == 1:
        return [0]
    return list(np.linspace(0, n_total - 1, n_sample).round().astype(int))

In [ ]:
import h5py

def dat_globber_multi(bl_len, bl_ang, pol_read, model_dir=None):
    
    print("pol_read ", pol_read)
    
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1'
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_pI/' 
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_xx/' 
    OUTPUT_DIR = f'output_vis_sigloss_check_out/Sig_Loss_{run_batch[0]}_{pol_read}/' 
    BASE_OUTDIR= Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/outputs")
    print("OUTPUT_DIR ", OUTPUT_DIR)
    
#     pol='pI'
#     pol='xx'
#     pol='yy'
    
#     proc1 = ["cutbl", "allbl"]
    
    # PTSRC SKY

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # ideal ENU, diameter Airy var (deltaD ~ <.2m)        airyprb, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # real ENU, diameter Airy var (deltaD ~ <.2m)         airyprb, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)      airytilt, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # EOR SKY =============================================

    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # Gaussian###################
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    
    # EOR SKY PLAYGROUND =============================================

    # Gaussian###################
    # MODEL_DIR  = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix_freqclone.03001951._beammapperant_airyred-nonred_airyred")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_spatmean_pwlw/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")

        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")

    # Airy ######################
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_tilt_skysd111_eoroffsetfix.06127f13._beammapperant_airytilt-nonred_airytilt/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")    
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy, CV rlzn challenge
    # MODEL_DIR is supplied by the caller (one per realization):
    MODEL_DIR  = Path(model_dir)
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # HERA Stripe Zenith Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_bright_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # HERA Stripe Zenith 5 Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_5_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # Isotropic ######################  
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    
    # EOR Noisy 

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-2x/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-correct-beam/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
         # Rlzn 556
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # PTSRC Noisy =============================================

    # PURE Noise =============================================

    # MODEL_DIR   = Path("noise-only-300k/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")

    # sky_type = "ptsrc"    
    sky_type = "eor_ns"
    
    DATA_PATH = BASE_OUTDIR / MODEL_DIR

    # Get just the last path component
    base = MODEL_DIR.name
    # Extract ideal tag: after "subset_" and before the next "_"
    m_ideal = re.search(r"subset_([^_]+)", base)
    ideal_tag = m_ideal.group(1) if m_ideal else "ideal"
    # Extract airy tag: after "nonred_" to the end (no more "_")
    m_airy = re.search(r"nonred_([^_]+)$", base)
    airy_tag = m_airy.group(1) if m_airy else "airy"
    print("ideal_tag:", ideal_tag)  
    print("airy_tag :", airy_tag) 

    batchnum = 0 
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck{chunk_min:05d}-{chunk_max:05d}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_fch{fch_min:04d}-{fch_max:04d}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Coh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))

    print(input_datfile_list)
    print("input_datfile_list ", input_datfile_list[0:10])
    
    print("bl_len, bl_ang ", bl_len, bl_ang)
    if not input_datfile_list:
        print(f'[skip] no PSPEC files found in {DATA_PATH}')
        return None
    uvps_dat_list = []
    start = time.time()
    for fname in input_datfile_list: 
#         print("fname ", fname)
        uvpsi = {}
        psci = PSpecContainer(filename=fname, mode='r', keep_open=False)
#         print(psci.groups() )
        uvpsi = psci.get_pspec('dset0', 'dset0_x_dset0' )
#         print("uvpsi ", uvpsi)
#         print( psci.groups )
        uvps_dat_list.append(uvpsi)
    end = time.time()
    print(end - start)
    
#     for fname in input_datfile_list: 
#         with h5py.File(fname, 'r') as f:
#             # List the top-level groups/datasets in the file
#             print("Top-level keys:", list(f.keys()))
#             print("f ", f)
            
#             uvps_dat_list.append(f)

            # Suppose there is a group called 'mygroup', you can access it like this:
#             if 'data_spw0' in f:
#                 mygroup = f['data_spw0']
#                 print(mygroup)
#                 # print("Keys in 'mygroup':", list(mygroup.keys()))


    print("uvps_dat_list ", uvps_dat_list[:] )
    # global uvp 
    print("Combining ___________________________________")
    # Indices to remove
    indices_to_remove = {} 
    new_list = [value for idx, value in enumerate(uvps_dat_list) if idx not in indices_to_remove]    
    uvp  = uvpspec.combine_uvpspec(new_list, merge_history=True, verbose=False)
    print( uvp.get_blpairs() )
    return uvp

#     print("uvps_dat_tot ", uvps_dat_tot)
    
    

In [ ]:
import h5py

def dat_globber_multi_avg(bl_len, bl_ang, pol_read, model_dir=None):
    
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1'
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_pI/' 
#     OUTPUT_DIR = '/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_xx/' 
#     OUTPUT_DIR = f'/home/herastore02-1/H6C_scratch_rchandra/Sig_Loss_1_{pol_read}/' 
    OUTPUT_DIR = f'output_vis_sigloss_check_out/Sig_Loss_{run_batch[0]}_{pol_read}/'
    BASE_OUTDIR= Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/outputs") 

    
#     pol='pI'
#     pol='xx'
#     pol='yy'
    
#     proc1 = ["cutbl", "allbl"]

    
    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # ideal ENU, diameter Airy var (deltaD ~ <.2m)        airyprb, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # real ENU, diameter Airy var (deltaD ~ <.2m)         airyprb, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyprb")
    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)      airytilt, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("ptsrc256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # EOR SKY

    # ideal ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # real ENU, tilt Airy var (deltZa ~ 2-3 degree)       airytilt, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airytilt")

    # Vivaldi
    # ideal ENU, ideal Vivaldi       vivaldired, idealT
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")
    # real ENU, ideal Vivaldi       vivaldired, idealF
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_vivaldired")

    # Gaussian###################
    # MODEL_DIR   = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    
    # EOR SKY PLAYGROUND =============================================

    # Gaussian###################
    # MODEL_DIR  = Path("eor-grf-256/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix_freqclone.03001951._beammapperant_airyred-nonred_airyred")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_spatmean_pwlw/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")

        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.29493475._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_gauss_skysd111_eoroffsetfix.03001951._beammapperant_airyred-nonred_airyred/")

    # Airy ######################
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
        # Rlzn 222
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_tilt_skysd111_eoroffsetfix.06127f13._beammapperant_airytilt-nonred_airytilt/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111_eoroffsetfix.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")    
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/fftvis_xcheck/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D14m_airy_nonspectral_80MHzref_eoroffsetfix.7be63f46._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556, 14m Airy, CV rlzn challenge
    # MODEL_DIR is supplied by the caller (one per realization):
    MODEL_DIR  = Path(model_dir)
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # HERA Stripe Zenith Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_bright_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # HERA Stripe Zenith 5 Pixel ######################
    # MODEL_DIR  = Path("eor-grf-256/zenith_5_point/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
    
    # Isotropic ######################  
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_111_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR  = Path("eor-grf-256/rlzn_seed_222_offsetfix/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_iso_skysd111_eoroffsetfix.920cf40b._beammapperant_airyred-nonred_airyred/")
    
    # EOR Noisy 

    # ideal ENU, ideal Airy                               airyred, idealT
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-2x/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-correct-beam/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_skysd111.5cfe1b35._beammapperant_airyred-nonred_airyred/")
    # real ENU, ideal Airy                                airyred, idealF
    # MODEL_DIR   = Path("eor-grf-256-noisy/nt17280-00288chunks-HERA_custom_subset_idealF_cba81417555edaffd87557575713cb61.txt-nonred_airyred")
         # Rlzn 556
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_nonspectral_80MHzref_eoroffsetfix.a74a7b9d._beammapperant_airyred-nonred_airyred/")
        # Rlzn 556 1 tilt
    # MODEL_DIR   = Path("eor-grf-256-noisy-D7m-airy/rlzn_seed_556_offsetfix_freqslic_grf/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_D7m_airy_1tilt_nonspectral_80MHzref_eoroffsetfix.378b3c2a._beammapperant_airytilt-nonred_airytilt/")        
    
    # PTSRC Noisy =============================================

    # PURE Noise =============================================

    # MODEL_DIR   = Path("noise-only-300k/nt17280-00288chunks-HERA_custom_subset_idealT_cba81417555edaffd87557575713cb61.txt.beam_map_10_airy.9105a7bf._beammapperant_airyred-nonred_airyred/")
    
    # sky_type = "ptsrc"    
    sky_type = "eor_ns"
    
    DATA_PATH = BASE_OUTDIR / MODEL_DIR

    # Get just the last path component
    base = MODEL_DIR.name
    # Extract ideal tag: after "subset_" and before the next "_"
    m_ideal = re.search(r"subset_([^_]+)", base)
    ideal_tag = m_ideal.group(1) if m_ideal else "ideal"
    # Extract airy tag: after "nonred_" to the end (no more "_")
    m_airy = re.search(r"nonred_([^_]+)$", base)
    airy_tag = m_airy.group(1) if m_airy else "airy"
    print("ideal_tag:", ideal_tag)  
    print("airy_tag :", airy_tag) 

    batchnum = 0 
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck{chunk_min:05d}-{chunk_max:05d}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_fch{fch_min:04d}-{fch_max:04d}_ck*_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))
    # input_datfile_list = sorted(glob.glob(os.path.join(DATA_PATH, f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5" )))

    print(f"notebook_{sky_type}_{ideal_tag}_{airy_tag}_Incoh_PSPEC_{bl_len}_{bl_ang}_{pol_read}_{proc1[0]}_{proc2[0]}_bch_{batchnum}_psc_PN.h5")

    print("input_datfile_list ", input_datfile_list[0:10])

    print("bl_len, bl_ang ", bl_len, bl_ang)
    
    if not input_datfile_list:
        print(f'[skip] no PSPEC files found in {DATA_PATH}')
        return None
    uvps_dat_list = []
    start = time.time()
    for fname in input_datfile_list: 
        uvpsi = {}
        psci = PSpecContainer(filename=fname, mode='r', keep_open=False)
#         print(psci.groups() )
        uvpsi = psci.get_pspec('dset0', 'dset0_x_dset0' )
#         print("uvpsi ", uvpsi)
#         print( psci.groups )
        uvps_dat_list.append(uvpsi)
    end = time.time()
    print(end - start)
    
#     for fname in input_datfile_list: 
#         with h5py.File(fname, 'r') as f:
#             # List the top-level groups/datasets in the file
#             print("Top-level keys:", list(f.keys()))
#             print("f ", f)
            
#             uvps_dat_list.append(f)

            # Suppose there is a group called 'mygroup', you can access it like this:
#             if 'data_spw0' in f:
#                 mygroup = f['data_spw0']
#                 print(mygroup)
#                 # print("Keys in 'mygroup':", list(mygroup.keys()))


#     print("uvps_dat_list ", uvps_dat_list)
    # global uvpspec_averaged 
    print("Combining ___________________________________")
    uvpspec_averaged  = uvpspec.combine_uvpspec(uvps_dat_list, merge_history=True, verbose=False)
    print( uvpspec_averaged.get_blpairs() )
    return uvpspec_averaged
    
#     print("uvps_dat_tot ", uvps_dat_tot)
    
    

In [ ]:
# ===========================================================================
# Build a LIST of combined-uvp dicts, one per realization (seed).
#   combined_uvp_dicts[r][(bl_len, bl_ang, pol)]      -> coherent  UVPSpec
#   combined_uvp_avg_dicts[r][(bl_len, bl_ang, pol)]  -> incoherent UVPSpec
# ===========================================================================
combined_uvp_dicts = []        # coherent,   one dict per realization
combined_uvp_avg_dicts = []    # incoherent, one dict per realization
REALIZATION_LABELS = []

for ri, md in enumerate(REALIZATIONS):
    lab = realization_label(md)
    print(f"\n================ realization {ri}: {lab} ================")
    d, da = {}, {}
    for pol_in in pol_root:
        for i in range(len(bl_len)):
            key = (bl_len[i], bl_ang[i], pol_in)
            uvp     = dat_globber_multi(bl_len[i], bl_ang[i], pol_in, model_dir=md)
            uvp_avg = dat_globber_multi_avg(bl_len[i], bl_ang[i], pol_in, model_dir=md)
            if uvp is not None:
                d[key] = uvp
            if uvp_avg is not None:
                da[key] = uvp_avg
    if d:                       # keep only realizations that actually loaded data
        combined_uvp_dicts.append(d)
        combined_uvp_avg_dicts.append(da)
        REALIZATION_LABELS.append(lab)
    else:
        print(f"[skip] realization {lab} has no PSPEC data; excluded.")

print(f"\nLoaded {len(combined_uvp_dicts)} realizations: {REALIZATION_LABELS}")
assert combined_uvp_dicts, "No realizations loaded -- check REALIZATIONS / run_batch / fch range."

# Back-compat aliases: the kept single-realization plot cells below operate on
# the FIRST realization via these names.
combined_uvp_dict = combined_uvp_dicts[0]
combined_uvp_avg_dict = combined_uvp_avg_dicts[0]

In [ ]:
# --- all 14 bands -------------------------------------------------
center_z_all = [
    24.6, 19.9, 16.8, 11.7, 10.8,  9.9,  8.9,
     8.2,  7.6,  7.1,  6.5,  6.0,  5.6,  5.2
]

avg_freq_all_MHz = [
   (50.2 +  62.2)/2,   #  56.20
   (63.3 +  73.5)/2,   #  68.40
   (74.6 +  85.4)/2,   #  80.00
  (108.0 + 116.1)/2,   # 112.05
  (117.3 + 124.4)/2,   # 120.85
  (125.4 + 136.2)/2,   # 130.80
  (138.3 + 148.2)/2,   # 143.25
  (150.1 + 159.2)/2,   # 154.65
  (159.3 + 169.9)/2,   # 164.60
  (171.9 + 181.1)/2,   # 176.50
  (181.4 + 196.4)/2,   # 188.90
  (198.5 + 208.4)/2,   # 203.45
  (212.3 + 220.6)/2,   # 216.45
  (224.3 + 231.1)/2    # 227.70
]

# --- only the 8 bands flagged “Used ✓” ----------------------------
center_z_used = [
    24.6, 19.9, 16.8, 10.8,  9.9,  7.6,  7.1,  5.6
]

avg_freq_used_MHz = [
   (50.2 +  62.2)/2,   #  56.20
   (63.3 +  73.5)/2,   #  68.40
   (74.6 +  85.4)/2,   #  80.00
  (117.3 + 124.4)/2,   # 120.85
  (125.4 + 136.2)/2,   # 130.80
  (159.3 + 169.9)/2,   # 164.60
  (171.9 + 181.1)/2,   # 176.50
  (212.3 + 220.6)/2    # 216.45
]


In [ ]:
# PLOT PSPEC 

# %matplotlib notebook
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from mpl_toolkits.axes_grid1 import make_axes_locatable

def bin_along_lst(lst_array, data, bin_width=0.5, stat="median"):
    """
    Bin a 1-D data array into uniform LST bins.

    Parameters
    ----------
    lst_array : array_like
        LST values in hours (same length as `data`).
    data : array_like
        The quantity to bin (e.g. direc_diff).
    bin_width : float
        Bin width in hours of LST (default 0.5 hr).
    stat : str
        Statistic per bin: 'mean', 'median', or 'both'.

    Returns
    -------
    bin_centers : ndarray
        Centre of each LST bin (hours).
    binned_vals : ndarray
        The chosen statistic in each bin (NaN where empty).
    binned_std  : ndarray
        Standard deviation in each bin (useful for errorbars).
    counts      : ndarray (int)
        Number of samples that fell in each bin.
    """
    lst_array = np.asarray(lst_array)
    data = np.asarray(data, dtype=float)

    # Build bin edges spanning the full LST range
    lst_min = np.nanmin(lst_array)
    lst_max = np.nanmax(lst_array)
    bin_edges = np.arange(lst_min, lst_max + bin_width, bin_width)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    nbins = len(bin_centers)
    binned_vals = np.full(nbins, np.nan)
    binned_std  = np.full(nbins, np.nan)
    counts      = np.zeros(nbins, dtype=int)

    # Digitize: assigns each LST sample to a bin index (1-based)
    indices = np.digitize(lst_array, bin_edges)  # 1..len(bin_edges)

    for i in range(nbins):
        mask = indices == (i + 1)          # digitize is 1-based
        n = np.count_nonzero(mask)
        counts[i] = n
        if n == 0:
            continue
        chunk = data[mask]
        if stat == "median":
            binned_vals[i] = np.nanmedian(chunk)
        elif stat == "mean":
            binned_vals[i] = np.nanmean(chunk)
        elif stat == "both":
            binned_vals[i] = np.nanmedian(chunk)   # store median; mean available via separate call
        binned_std[i] = np.nanstd(chunk)

    return bin_centers, binned_vals, binned_std, counts

def get_spw_info(uvp):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    spw_indices = uvp.spw_array   # e.g., shape (14,)
    freq = uvp.freq_array         # e.g., shape (1114,)
    spw_freq = uvp.spw_freq_array # e.g., shape (1114,)
    
    spw_ranges = {}
    for spw in spw_indices:
        mask = (spw_freq == spw)
        if np.any(mask):
            freq_min = np.min(freq[mask])
            freq_max = np.max(freq[mask])
            spw_ranges[spw] = (freq_min, freq_max)
        else:
            spw_ranges[spw] = None
    return spw_ranges

def get_plot_data(uvp, uvpspec_averaged, pol, idx):
    """
    Returns a dictionary mapping each spw index to its (min, max) frequency range (in Hz).
    """
    blp = uvp.get_blpairs()[0]
    # key = (idx, blp, 'xx')
    key = (idx, blp, pol)

    # Retrieve LST array and convert to hours.
    tarr = uvp.lst_avg_array
    tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
    lst_array_rad = tarr
#         print("lst_array_rad ", lst_array_rad)
    lst_array = tarrq

    dlys = uvp.get_dlys(idx) * 1e9
    index_of_zero = np.where(np.isclose(dlys, 0))[0][0]

    uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
    uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]

    percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
    median_val = np.nanmedian(percent_diff)
    median_vals.append(median_val)
    mean_val = np.nanmean(percent_diff)
    mean_vals.append(mean_val)

    lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        
    return dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals


def get_plot_data_all_blpairs(uvp, uvp_avg, pol, spw):
    """
    Gather τ≈0 spectra from *all* baseline–pairs, concatenate by LST,
    and compute percent-difference between uvp and its averaged version.

    Parameters
    ----------
    uvp, uvp_avg : UVPSpec
        Original and incoherently-averaged spectra.
    pol          : str              (e.g. 'xx' or 'pI')
    spw          : int              (spectral-window index)

    Returns
    -------
    dlys               : (Ndlys,)  delay values [ns]
    zero_idx           : int       index of τ≈0 in `dlys`
    lst_rad_sorted     : (Ntimes,) LST in radians, sorted
    lst_hr_sorted      : (Ntimes,) LST in hours,  "
    uvp_pow_sorted     : (Ntimes,) |P|  from uvp          (τ≈0)
    uvp_avg_pow_sorted : (Ntimes,) |P|  from uvp_avg      (τ≈0)
    pct_diff_sorted    : (Ntimes,) 100*(uvp-avg)/avg
    mean_pct           : float      mean of pct_diff
    median_pct         : float      median of pct_diff
    """
    # ---------- fixed per-spw info ----------
    dlys = uvp.get_dlys(spw) * 1e9          # ns
    print("dlys ", dlys)
    print( (np.where(np.isclose(dlys, 558.54545455))) )
    zero_idx = int(np.where(np.isclose(dlys, 0))[0][0])     # 1024

    # ---------- gather blocks ----------
    lst_list         = []
    uvp_pow_list     = []
    uvp_avg_pow_list = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        print("key ", key)

        # LST (radians) and convert now (same for both uvp and uvp_avg)
        lst_block = uvp.lst_avg_array[uvp.blpair_to_indices(blp)]
        lst_list.append(lst_block)

        # spectra, pick τ≈0 and |.| for power
        uvp_pow_block     = np.abs(np.real(uvp        .get_data(key)))[:, zero_idx]
        print("uvp_pow_block ", uvp_pow_block)
        uvp_avg_pow_block = np.abs(np.real(uvp_avg    .get_data(key)))[:, zero_idx]
        print("uvp_avg_pow_block ", uvp_avg_pow_block)

        uvp_pow_list    .append(uvp_pow_block)
        uvp_avg_pow_list.append(uvp_avg_pow_block)

    # ---------- concatenate and sort by LST ----------
    lst_all         = np.concatenate(lst_list)
    uvp_pow_all     = np.concatenate(uvp_pow_list)
    uvp_avg_pow_all = np.concatenate(uvp_avg_pow_list)

    order           = np.argsort(lst_all)
    lst_rad_sorted  = lst_all        [order]
    uvp_pow_sorted  = uvp_pow_all    [order]
    uvp_avg_sorted  = uvp_avg_pow_all[order]

    # ---------- compute percent difference ----------
    pct_diff_sorted = 100.0 * (uvp_pow_sorted - uvp_avg_sorted) / np.where(
                         uvp_avg_sorted != 0, uvp_avg_sorted, np.nan
                     )

    mean_pct   = np.nanmean(pct_diff_sorted)
    median_pct = np.nanmedian(pct_diff_sorted)

    # LST in hours
    lst_hr_sorted = lst_rad_sorted * (12 / np.pi)
    
    lst_array_roll = np.where(lst_hr_sorted > 20, lst_hr_sorted - 24, lst_hr_sorted)

    return (dlys, zero_idx,
            lst_rad_sorted, lst_hr_sorted, lst_array_roll,
            uvp_pow_sorted, uvp_avg_sorted,
            pct_diff_sorted, mean_pct, median_pct)


median_vals = []
mean_vals = []

def plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict, zero_delay_pspec_spw=None, avg_pspec=None):
    """
    Plot percent difference between uvp and uvpspec_averaged versus LST for each SPW.
    The title and legend include:
      - The baseline length (in m)
      - The baseline angle (forced to 0° here)
      - The SPW index and its full frequency range (in MHz)
      - The polarization used.
    """
    # for grp_key in combined_uvp_dict:
    #     print("grp_key ", grp_key, grp_key[2])
    #     pol_in=grp_key[2]
    #     spw_ranges = get_spw_info(combined_uvp_dict[grp_key])
    example_uvp = next(iter(combined_uvp_dict.values()))
    spw_ranges = get_spw_info(example_uvp)
    print("spw_ranges ", spw_ranges)
    num_spws = len(spw_ranges)
    fig, axes = plt.subplots(5, 3, figsize=(30, 18), sharex=False)
    axes = axes.flatten(order='C')
    
#     pol = 'pI'
    pol = 'xx'

    
    # Compute baseline information.
    # If you expect a horizontal baseline, you can force the angle to 0°.
    bl_vec = example_uvp.bl_vecs[0]  # using the first baseline vector as an example
    print("bl_vec ", bl_vec)
    baseline_length = np.linalg.norm(bl_vec)  # in meters
    baseline_angle = 0  # Force to 0° (if that's what you expect)
    # Otherwise, if you want to compute the angle:
    # baseline_angle = (np.arctan2(bl_vec[1], bl_vec[0]))
    
    # Iterate over SPWs using sorted items so we can unpack the (min, max) frequency tuple.
    for idx, (spw_key, spw_val) in enumerate(sorted(spw_ranges.items(), key=lambda x: x[0])):
        if spw_val is not None:
            freq_min, freq_max = spw_val
            # Create a string showing the full frequency range in MHz.
            freq_range_str = f"{freq_min/1e6:.2f}-{freq_max/1e6:.2f} MHz"
        else:
            freq_range_str = "N/A"
        
        # (Optional) If you have zero_delay_pspec_spw and avg_pspec available, use them:
        if zero_delay_pspec_spw is not None:
            bl = redgrp_unpol_comb[0]  # e.g., baseline pair (3,5)
            zero_delay_power = np.abs(zero_delay_pspec_spw[bl][pol][:, idx])
            avg_power = avg_pspec[idx][pol]
        
        if zero_delay_pspec_spw is not None:
            percent_diff_totpow = 100 * (zero_delay_power - avg_power) / np.where(avg_power != 0, avg_power, np.nan)
        
        blp = example_uvp.get_blpairs()[0]
        key = (idx, blp, 'xx')
        
#         for i in range(1):#len(bl_len)):
        for grp_key in combined_uvp_dict:
            print("grp_key ", grp_key)
            dlys, index_of_zero, lst_array_rad, lst_array, lst_array_roll, uvp_power, uvpspec_averaged_power, percent_diff, mean_vals, med_vals = get_plot_data_all_blpairs(combined_uvp_dict[grp_key], combined_uvp_avg_dict[grp_key], pol, idx)
        print( lst_array_roll.shape, 
              percent_diff.shape
             )
        
        # Retrieve LST array and convert to hours.
#         tarr = uvp.lst_avg_array
#         tarrq = tarr * (12 / np.pi)  # Convert from radians to hours.
#         lst_array_rad = tarr
# #         print("lst_array_rad ", lst_array_rad)
#         lst_array = tarrq
        
#         dlys = uvp.get_dlys(idx) * 1e9
#         index_of_zero = np.where(np.isclose(dlys, 0))[0][0]
        
#         uvp_power = np.abs(np.real(uvp.get_data(key)))[:, index_of_zero]
#         uvpspec_averaged_power = np.abs(np.real(uvpspec_averaged.get_data(key)))[:, index_of_zero]
        
        # percent_diff = 100 * (uvp_power - uvpspec_averaged_power) / np.where(uvpspec_averaged_power != 0, uvpspec_averaged_power, np.nan)
        direc_diff = uvp_power - uvpspec_averaged_power
        print("percent_diff shape ", percent_diff.shape)
#         median_val = np.nanmedian(percent_diff)
#         median_vals.append(median_val)
#         mean_val = np.nanmean(percent_diff)
#         mean_vals.append(mean_val)

        # sine^2 with period = 1 hour LST
        sin2_lst = np.sin(np.pi * lst_array_roll) ** 2
        amplitude = 5000.0  # adjust as needed
        sin2_signal = (amplitude * np.sin(np.pi * lst_array_roll) ** 2) + 100
        
        lst_array_roll = np.where(lst_array > 20, lst_array - 24, lst_array)
        ax = axes[idx]
        ax.axhline(10, color='blue', alpha=0.5)
        ax.axhline(-10, color='blue', alpha=0.5)
        ax.axhline(5, color='green', alpha=0.5)
        ax.axhline(-5, color='green', alpha=0.5)
        ax.axhline(0, color='black', alpha=0.5)
        
        # Scatter plot: percent difference vs. LST (rolled), using a softer "skyblue" color.
        # ax.plot(lst_array_roll, sin2_signal, color="orange", linewidth=1.5, label=r"$\sin^2(\pi \cdot \mathrm{LST})$, T=1hr")
        # ax.scatter(lst_array_roll, direc_diff, s=1, marker="o", linestyle="-", color="royalblue",
        #            label=f"PSPEC SPW {freq_range_str}")
        ax.scatter(lst_array_roll, uvpspec_averaged_power, s=1, marker="o", linestyle="-", color="royalblue",
                   label=f"PSPEC SPW {freq_range_str}")
        ax.scatter(lst_array_roll, uvp_power, s=1, marker="o", linestyle="-", color="red",
                   label=f"PSPEC SPW {freq_range_str}", alpha = 0.2)
        # Bin with 0.25-hour LST bins, using median
        # bin_cen, bin_val, bin_err, bin_n = bin_along_lst(lst_array_roll, direc_diff, bin_width=0.25, stat="median")
        # # Overlay on the existing axis
        # ax.errorbar(bin_cen, bin_val, yerr=bin_err, fmt="o-", color="orange",
        #             markersize=4, linewidth=1.2, capsize=2,
        #             label=f"Binned (Δt={0.25} hr, median)")
        
        ax.xaxis.set_major_locator(MultipleLocator(1))
        # ax.set_ylim(-20, 20)
        ax.set_xlim(-4, 8)
        # ax.set_ylim(1e6, 2e9)
        ax.set_xlim(-4,20)
        ax.set_yscale('log')
        ax.grid(False)
        ax.set_ylabel(r"PSPEC [mK$^2$]", fontsize=12)
        ax.set_xlabel("LST (Hours)", fontsize=12)
        
        print("np.max(direc_diff) ", np.nanmax(np.abs(direc_diff)))
        print("np.max(direc_diff) ", np.nanmin(np.abs(direc_diff)))
#         print(">0% spw, LST, % ", idx, len(lst_array[direc_diff==np.nanmax(direc_diff)]), lst_array[direc_diff==np.nanmax(direc_diff)], '\n LST rad', lst_array_rad[direc_diff==np.nanmax(direc_diff)], '\n', direc_diff[direc_diff==np.nanmax(direc_diff)] )
#         print("in -5-6% spw, LST, % ", idx, len(lst_array[(direc_diff > -6) & (direc_diff < -5)]), lst_array[(direc_diff > -6) & (direc_diff < -5)], '\n LST rad', lst_array_rad[(direc_diff > -6) & (direc_diff < -5)], '\n', direc_diff[(direc_diff > -6) & (direc_diff < -5)] )
        
        if pol == 'nn':
            pol1 = 'xx'
        else:
            pol1 = pol
        
        # Set title with baseline length, forced angle, SPW index, frequency range, and polarization.
        ax.set_title(f"Len {int(baseline_length)}m, Ang {int(baseline_angle)}°, SPW {idx}, Freq: {freq_range_str}, Pol: {pol1}", fontsize=14)
        ax.legend(fontsize=10)
    
    # Hide any unused subplots.
    for idx in range(len(spw_ranges), len(axes)):
        fig.delaxes(axes[idx])
    
    plt.tight_layout()
    plt.show()
    
#     print("Median percent difference for each SPW:")
#     for spw_key, med in zip(sorted(spw_ranges.keys()), median_vals):
#         print(f"SPW {spw_key}: median % difference = {med:.2f}%")
#     print("Mean percent difference for each SPW:")
#     for spw_key, mea in zip(sorted(spw_ranges.keys()), mean_vals):
#         print(f"SPW {spw_key}: mean % difference = {mea:.2f}%")

# Example usage:
plot_percent_difference(combined_uvp_dict, combined_uvp_avg_dict)


## 0-delay PSPEC vs LST -- all realizations overlaid

In [ ]:
# ===========================================================================
# 0-delay PSPEC vs LST -- ALL realizations (seeds) overlaid on one plot.
# Reuses get_plot_data_all_blpairs / get_spw_info defined above.
# ===========================================================================
import numpy as np
import matplotlib.pyplot as plt


def plot_zero_delay_overlay(combined_uvp_dicts, combined_uvp_avg_dicts, labels,
                            group_key=None, spw=None, pol="xx", which="both"):
    """Overlay the tau~0 PSPEC-vs-LST track for every realization (seed).

    which : "both" (coherent + incoherent), "coherent", or "incoherent".
    """
    d0 = combined_uvp_dicts[0]
    if group_key is None:
        group_key = list(d0.keys())[-1]
    uvp0 = d0[group_key]
    spw_info = get_spw_info(uvp0)
    if spw is None:
        valid = [k for k, v in sorted(spw_info.items()) if v is not None]
        spw = valid[0] if valid else sorted(spw_info)[0]
    fr = spw_info.get(spw)
    fr_str = f"{fr[0]/1e6:.2f}-{fr[1]/1e6:.2f} MHz" if fr else "N/A"
    bl_m = float(np.linalg.norm(uvp0.bl_vecs[0]))

    fig, ax = plt.subplots(figsize=(11.5, 5.8), constrained_layout=True)
    cmap = plt.get_cmap("viridis", max(2, len(combined_uvp_dicts)))
    
    _lst_ref = None                                             
    _coh_stack, _incoh_stack = [], []

    for r, (dd, da) in enumerate(zip(combined_uvp_dicts, combined_uvp_avg_dicts)):
        if group_key not in dd or group_key not in da:
            continue
        (_, _, _, _, lst_roll,
         uvp_pow, uvp_avg_pow, _, _, _) = get_plot_data_all_blpairs(
            dd[group_key], da[group_key], pol, spw)
         
        if _lst_ref is None:
            _lst_ref = lst_roll
        if uvp_pow.shape == _lst_ref.shape:          # only stack seeds on the same LST grid
            _coh_stack.append(uvp_pow)
            _incoh_stack.append(uvp_avg_pow)
 
         
        c = cmap(r)
        lab = labels[r] if r < len(labels) else f"r{r}"
        if which in ("both", "coherent"):
            ax.scatter(lst_roll, uvp_pow, s=5, color=c, alpha=0.85, label=f"{lab} (coh)")
        if which in ("both", "incoherent"):
            ax.scatter(lst_roll, uvp_avg_pow, s=5, color=c, marker="x", alpha=0.5,
                       label=f"{lab} (incoh)")
            
    if _coh_stack:
        coh_mean   = np.nanmean(np.vstack(_coh_stack),   axis=0)   # ensemble-mean power (linear)
        incoh_mean = np.nanmean(np.vstack(_incoh_stack), axis=0)
        order = np.argsort(_lst_ref)                                # clean line across the 24h wrap
        if which in ("both", "coherent"):
            ax.plot(_lst_ref[order], coh_mean[order], color="pink", lw=4.0,
                    zorder=5, label=f"seed mean (coh, N={len(_coh_stack)})")
        if which in ("both", "incoherent"):
            ax.plot(_lst_ref[order], incoh_mean[order], color="red", lw=4.0,
                    zorder=5, label=f"seed mean (incoh, N={len(_incoh_stack)})")

            
    ax.set_yscale("log")
    ax.set_xlabel("LST (hours)")
    ax.set_ylabel(r"PSPEC ($\tau\approx 0$) [mK$^2$]")
    ax.set_title(f"0-delay PSPEC vs LST -- all realizations | "
                 f"b={bl_m:.2f} m, SPW {spw} ({fr_str}), pol={pol}")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, ncol=2, loc="best")
    plt.show()
    return {"group_key": group_key, "spw": spw, "baseline_m": bl_m}


zero_delay_overlay_info = plot_zero_delay_overlay(
    combined_uvp_dicts, combined_uvp_avg_dicts, REALIZATION_LABELS, pol=pol_root[0])

In [ ]:
# Fourier spectrum along the LST axis for the PSPEC curves plotted above

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows as signal_windows


def get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw):
    """
    Recompute the same tau ~= 0 LST tracks plotted in the previous PSPEC cell,
    but without the diagnostic print statements.
    """
    dlys = uvp.get_dlys(spw) * 1e9
    zero_candidates = np.where(np.isclose(dlys, 0.0))[0]
    zero_idx = int(zero_candidates[0]) if zero_candidates.size else int(np.nanargmin(np.abs(dlys)))

    lst_blocks = []
    uvp_power_blocks = []
    uvp_avg_power_blocks = []

    for blp in uvp.get_blpairs():
        key = (spw, blp, pol)
        lst_blocks.append(uvp.lst_avg_array[uvp.blpair_to_indices(blp)])
        uvp_power_blocks.append(np.abs(np.real(uvp.get_data(key)))[:, zero_idx])
        uvp_avg_power_blocks.append(np.abs(np.real(uvp_avg.get_data(key)))[:, zero_idx])

    lst_rad = np.concatenate(lst_blocks)
    lst_hr = lst_rad * (12.0 / np.pi)
    lst_array_roll = np.where(lst_hr > 20.0, lst_hr - 24.0, lst_hr)
    uvp_power = np.concatenate(uvp_power_blocks)
    uvpspec_averaged_power = np.concatenate(uvp_avg_power_blocks)

    order = np.argsort(lst_array_roll)
    return (
        dlys,
        zero_idx,
        lst_rad[order],
        lst_hr[order],
        lst_array_roll[order],
        uvp_power[order],
        uvpspec_averaged_power[order],
    )


def _uniform_lst_grid(lst_hours, values, dt_hours=None, statistic="median"):
    """Bin/interpolate an LST series onto a uniform grid for an FFT."""
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]

    if x.size < 2:
        raise ValueError("Need at least two finite LST samples for an FFT.")

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    unique_x = np.unique(x)
    positive_dx = np.diff(unique_x)
    positive_dx = positive_dx[positive_dx > 0]
    if dt_hours is None:
        if positive_dx.size == 0:
            raise ValueError("Could not infer an LST sampling interval.")
        dt_hours = np.nanmedian(positive_dx)

    if not np.isfinite(dt_hours) or dt_hours <= 0:
        raise ValueError("dt_hours must be a positive finite number.")

    n_grid = int(np.floor((x.max() - x.min()) / dt_hours)) + 1
    lst_grid = x.min() + dt_hours * np.arange(n_grid)
    if lst_grid[-1] < x.max() - 0.25 * dt_hours:
        lst_grid = np.append(lst_grid, lst_grid[-1] + dt_hours)

    y_grid = np.full(lst_grid.size, np.nan)
    bin_index = np.floor((x - (lst_grid[0] - 0.5 * dt_hours)) / dt_hours).astype(int)
    valid = (bin_index >= 0) & (bin_index < lst_grid.size)

    for idx in np.unique(bin_index[valid]):
        chunk = y[valid & (bin_index == idx)]
        if statistic == "mean":
            y_grid[idx] = np.nanmean(chunk)
        else:
            y_grid[idx] = np.nanmedian(chunk)

    good = np.isfinite(y_grid)
    if np.count_nonzero(good) < 2:
        raise ValueError("Need at least two populated LST bins for an FFT.")

    if not np.all(good):
        y_grid = np.interp(lst_grid, lst_grid[good], y_grid[good])

    return lst_grid, y_grid, dt_hours


def _clean_sort_lst_samples(lst_hours, values):
    """Return finite LST/value samples sorted by LST."""
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=float)
    finite = np.isfinite(x) & np.isfinite(y)
    x = x[finite]
    y = y[finite]
    if x.size < 2:
        raise ValueError("Need at least two finite LST samples for a Fourier transform.")
    order = np.argsort(x)
    return x[order], y[order]


def _collapse_repeated_lst_samples(lst_hours, values, dt_hours=None, statistic="median", duplicate_tol=None):
    """
    Collapse repeated or nearly repeated LST samples before a Fourier transform.

    This is useful because concatenating many redundant baseline-pair PSPEC values can
    produce several values at the same LST. Treating those repeats as a time stream
    would overweight that LST, so by default we reduce each repeated-LST group to
    one robust representative value.
    """
    x, y = _clean_sort_lst_samples(lst_hours, values)

    unique_x = np.unique(x)
    positive_dx = np.diff(unique_x)
    positive_dx = positive_dx[positive_dx > 0]
    if duplicate_tol is None:
        if dt_hours is not None and np.isfinite(dt_hours) and dt_hours > 0:
            duplicate_tol = 0.05 * dt_hours
        elif positive_dx.size:
            duplicate_tol = 0.05 * np.nanmedian(positive_dx)
        else:
            duplicate_tol = 0.0

    groups_x = []
    groups_y = []
    start = 0
    for idx in range(1, x.size + 1):
        at_end = idx == x.size
        starts_new_group = False if at_end else (x[idx] - x[start] > duplicate_tol)
        if at_end or starts_new_group:
            x_chunk = x[start:idx]
            y_chunk = y[start:idx]
            groups_x.append(np.nanmedian(x_chunk))
            if statistic == "mean":
                groups_y.append(np.nanmean(y_chunk))
            else:
                groups_y.append(np.nanmedian(y_chunk))
            start = idx

    return np.asarray(groups_x, dtype=float), np.asarray(groups_y, dtype=float)


def _is_uniform_sampling(lst_hours, dt_hours=None, rtol=1e-5, atol=1e-8):
    """Check whether sorted LST samples are regularly spaced to tolerance."""
    x = np.asarray(lst_hours, dtype=float)
    if x.size < 2:
        return False, np.nan
    dx = np.diff(x)
    positive_dx = dx[dx > 0]
    if positive_dx.size == 0:
        return False, np.nan
    dt = np.nanmedian(positive_dx) if dt_hours is None else dt_hours
    if not np.isfinite(dt) or dt <= 0:
        return False, np.nan
    tol = max(float(atol), float(rtol) * abs(dt))
    return bool(np.all(np.abs(dx - dt) <= tol)), dt


def _lst_window_weights(n, window):
    """Construct a Fourier taper using NumPy/SciPy windows where available."""
    window_key = None if window is None else str(window).lower().replace("_", "-")
    if window_key == "hann" and n >= 3:
        return np.hanning(n)
    if window_key in ("blackmanharris", "blackman-harris", "blackman harris", "bh") and n >= 3:
        return signal_windows.blackmanharris(n, sym=False)
    if window_key in (None, "boxcar", "rect", "rectangular"):
        return np.ones(n)
    raise ValueError("window must be 'hann', 'blackmanharris', 'boxcar', or None.")


def _preprocess_lst_fft_values(values, normalize="fractional", remove_mean=True):
    """Apply optional fractional normalization and optional DC/mean removal."""
    y_fft = np.asarray(values, dtype=float).copy()
    normalize_key = "none" if normalize is None else str(normalize).lower()

    if normalize_key == "fractional":
        scale = np.nanmedian(np.abs(y_fft))
        if np.isfinite(scale) and scale > 0:
            y_fft = y_fft / scale - 1.0
    elif normalize_key in ("none", "raw"):
        pass
    else:
        raise ValueError("normalize must be 'fractional', 'none', 'raw', or None.")

    if remove_mean:
        y_fft = y_fft - np.nanmean(y_fft)
    return np.nan_to_num(y_fft, nan=0.0, posinf=0.0, neginf=0.0)


def _one_sided_amplitude(fft, n, coherent_gain):
    """Convert an rFFT-like complex spectrum to a one-sided amplitude spectrum."""
    amp = np.abs(fft) / (n * coherent_gain)
    if n % 2 == 0 and amp.size > 2:
        amp[1:-1] *= 2.0
    elif n % 2 == 1 and amp.size > 1:
        amp[1:] *= 2.0
    return amp


def _direct_nonuniform_fourier(lst_hours, values, dt_hours=None, freq_grid=None):
    """
    Directly evaluate sum_n values[n] exp(-2 pi i f (t_n - t_0)).

    This avoids interpolation/gridding for irregular LST samples. It is slower than
    an FFT, but the arrays here are small enough that the clarity is useful.
    """
    x = np.asarray(lst_hours, dtype=float)
    y = np.asarray(values, dtype=complex)
    n = x.size
    if freq_grid is None:
        _, dt_eff = _is_uniform_sampling(x, dt_hours=dt_hours)
        if not np.isfinite(dt_eff) or dt_eff <= 0:
            positive_dx = np.diff(np.unique(x))
            positive_dx = positive_dx[positive_dx > 0]
            if positive_dx.size == 0:
                raise ValueError("Could not infer a frequency grid for the direct FT.")
            dt_eff = np.nanmedian(positive_dx)
        freq = np.fft.rfftfreq(n, d=dt_eff)
    else:
        freq = np.asarray(freq_grid, dtype=float)

    phase = np.exp(-2j * np.pi * freq[:, None] * (x[None, :] - x[0]))
    return freq, phase @ y


def lst_fft_spectrum(lst_hours,
                     values,
                     dt_hours=None,
                     normalize="fractional",
                     window="hann",
                     method="auto",
                     statistic="median",
                     collapse_repeats=True,
                     duplicate_tol=None,
                     remove_mean=True,
                     uniform_rtol=1e-5,
                     uniform_atol=1e-8,
                     freq_grid=None,
                     return_info=False):
    """
    Fourier transform a PSPEC-vs-LST track.

    Frequencies are in cycles per LST hour.

    Parameters
    ----------
    method : {'auto', 'grid', 'fft', 'direct', 'nonuniform'}
        'grid' reproduces the original robust notebook behavior: bin/interpolate onto
        a uniform grid and FFT. 'fft' requires the LST samples to be uniform after the
        optional repeated-LST collapse. 'direct'/'nonuniform' evaluates the Fourier
        sum at the requested frequencies without gridding. 'auto' uses an FFT when
        samples are already uniform and otherwise falls back to the direct sum.
    normalize : {'fractional', 'none', 'raw', None}
        'fractional' transforms P/median(|P|) - 1. Use None/'none'/'raw' to transform
        the PSPEC values themselves.
    remove_mean : bool
        If True, subtract the arithmetic mean before transforming. Set False to keep
        the DC mode and make the transform as raw as possible.
    window : {None, 'boxcar', 'hann', 'blackman-harris', 'blackmanharris', 'bh'}
        Optional LST taper. Blackman-Harris uses scipy.signal.windows.blackmanharris.
    """
    method_key = "auto" if method is None else str(method).lower()
    method_aliases = {"nudft": "direct", "nonuniform": "direct", "non-uniform": "direct"}
    method_key = method_aliases.get(method_key, method_key)
    if method_key not in ("auto", "grid", "fft", "direct"):
        raise ValueError("method must be 'auto', 'grid', 'fft', 'direct', or 'nonuniform'.")

    if method_key == "grid":
        lst_grid, y_grid, dt_eff = _uniform_lst_grid(
            lst_hours,
            values,
            dt_hours=dt_hours,
            statistic=statistic,
        )
        transform_method = "grid-fft"
    else:
        if collapse_repeats:
            lst_grid, y_grid = _collapse_repeated_lst_samples(
                lst_hours,
                values,
                dt_hours=dt_hours,
                statistic=statistic,
                duplicate_tol=duplicate_tol,
            )
        else:
            lst_grid, y_grid = _clean_sort_lst_samples(lst_hours, values)

        is_uniform, dt_eff = _is_uniform_sampling(
            lst_grid,
            dt_hours=dt_hours,
            rtol=uniform_rtol,
            atol=uniform_atol,
        )

        if method_key == "fft" and not is_uniform:
            raise ValueError("method='fft' requires uniformly sampled LSTs. Use method='grid', 'direct', or 'auto'.")
        if method_key == "auto":
            transform_method = "direct" if not is_uniform else "fft"
        else:
            transform_method = method_key

    y_fft = _preprocess_lst_fft_values(y_grid, normalize=normalize, remove_mean=remove_mean)
    n = y_fft.size
    weights = _lst_window_weights(n, window)
    coherent_gain = np.mean(weights)
    if not np.isfinite(coherent_gain) or coherent_gain == 0:
        coherent_gain = 1.0

    if transform_method in ("fft", "grid-fft"):
        fft = np.fft.rfft(y_fft * weights)
        freq = np.fft.rfftfreq(n, d=dt_eff)
    else:
        freq, fft = _direct_nonuniform_fourier(
            lst_grid,
            y_fft * weights,
            dt_hours=dt_eff,
            freq_grid=freq_grid,
        )

    amp = _one_sided_amplitude(fft, n, coherent_gain)
    info = {
        "method": transform_method,
        "requested_method": method_key,
        "dt_hours": dt_eff,
        "n_samples": int(n),
        "collapse_repeats": bool(collapse_repeats),
        "normalize": normalize,
        "remove_mean": bool(remove_mean),
        "window": window,
        "coherent_gain": coherent_gain,
    }

    if return_info:
        return freq, amp, lst_grid, y_grid, info
    return freq, amp, lst_grid, y_grid


def _running_percentile(values, window, percentile=50):
    """Simple centered running percentile; keeps this cell independent of scipy."""
    y = np.asarray(values, dtype=float)
    n = y.size
    if n == 0:
        return y

    window = int(window)
    if window < 3:
        window = 3
    if window % 2 == 0:
        window += 1
    if window > n:
        window = n if n % 2 else max(1, n - 1)

    half = window // 2
    out = np.full(n, np.nan)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        out[i] = np.nanpercentile(y[lo:hi], percentile)
    return out


def spectral_information_filter(freq,
                                amp,
                                n_features=5,
                                continuum_window=19,
                                continuum_percentile=45,
                                score_threshold=2.5,
                                min_frequency=0.0,
                                max_frequency=6.0,
                                min_separation_bins=2):
    """
    Pick Fourier feature regions by spectral whitening rather than by a fixed bandpass.

    Formalism:
      1. Work in log-amplitude, log10 A(f).
      2. Estimate a smooth continuum C(f) with a broad running percentile.
         This continuum captures the expected red/noisy fall from low to high cycles.
      3. Score each mode by its robust residual above the continuum:

             score(f) = [log10 A(f) - C(f)] / [1.4826 * MAD(residual)]

      4. Group contiguous above-threshold bins into feature regions and rank them
         by integrated positive excess. This makes the selection favor coherent
         bumps over isolated noisy high-frequency bins.
    """
    freq = np.asarray(freq, dtype=float)
    amp = np.asarray(amp, dtype=float)

    continuum = np.full_like(amp, np.nan, dtype=float)
    score = np.full_like(amp, np.nan, dtype=float)
    excess_ratio = np.full_like(amp, np.nan, dtype=float)

    valid = np.isfinite(freq) & np.isfinite(amp) & (freq > 0) & (amp > 0)
    if max_frequency is not None:
        valid &= freq <= max_frequency
    valid &= freq >= min_frequency

    valid_idx = np.flatnonzero(valid)
    if valid_idx.size < 5:
        return {
            "continuum": continuum,
            "score": score,
            "excess_ratio": excess_ratio,
            "features": [],
        }

    f_valid = freq[valid_idx]
    log_amp = np.log10(amp[valid_idx])
    log_continuum = _running_percentile(log_amp, continuum_window, percentile=continuum_percentile)
    residual = log_amp - log_continuum

    # A tiny median smooth suppresses one-bin spikes but preserves broad bumps.
    residual_for_peaks = _running_percentile(residual, 3, percentile=50)
    med = np.nanmedian(residual_for_peaks)
    mad = 1.4826 * np.nanmedian(np.abs(residual_for_peaks - med))
    if not np.isfinite(mad) or mad <= 0:
        mad = np.nanstd(residual_for_peaks)
    if not np.isfinite(mad) or mad <= 0:
        mad = 1.0

    local_score = (residual_for_peaks - med) / mad
    continuum[valid_idx] = 10.0 ** log_continuum
    score[valid_idx] = local_score
    excess_ratio[valid_idx] = 10.0 ** residual

    df = np.nanmedian(np.diff(f_valid)) if f_valid.size > 1 else 0.0
    half_df = 0.5 * df if np.isfinite(df) and df > 0 else 0.0

    def build_feature(region_local_idx):
        region_local_idx = np.asarray(region_local_idx, dtype=int)
        peak_loc = region_local_idx[np.nanargmax(local_score[region_local_idx])]
        peak_glob = valid_idx[peak_loc]
        region_glob = valid_idx[region_local_idx]

        weights = np.clip(residual[region_local_idx], 0.0, None)
        if np.nansum(weights) > 0:
            centroid = np.nansum(f_valid[region_local_idx] * weights) / np.nansum(weights)
        else:
            centroid = freq[peak_glob]

        region_score = np.nansum(np.clip(local_score[region_local_idx] - score_threshold, 0.0, None))
        if not np.isfinite(region_score) or region_score <= 0:
            region_score = np.nanmax(local_score[region_local_idx])

        f_min = max(0.0, freq[region_glob[0]] - half_df)
        f_max = freq[region_glob[-1]] + half_df

        return {
            "frequency_cyc_per_hr": freq[peak_glob],
            "frequency_centroid_cyc_per_hr": centroid,
            "frequency_min_cyc_per_hr": f_min,
            "frequency_max_cyc_per_hr": f_max,
            "period_hr": np.inf if freq[peak_glob] == 0 else 1.0 / freq[peak_glob],
            "amplitude": amp[peak_glob],
            "continuum": continuum[peak_glob],
            "excess_ratio": excess_ratio[peak_glob],
            "score": score[peak_glob],
            "region_score": region_score,
            "n_bins": int(region_local_idx.size),
        }

    above = np.isfinite(local_score) & (local_score >= score_threshold)
    regions = []
    start = None
    for i, is_above in enumerate(above):
        if is_above and start is None:
            start = i
        if start is not None and ((not is_above) or i == above.size - 1):
            stop = i if not is_above else i + 1
            regions.append(np.arange(start, stop))
            start = None

    if not regions:
        # Fallback: report the strongest positive local maxima if the threshold is too strict.
        local_max = np.zeros(valid_idx.size, dtype=bool)
        if valid_idx.size > 2:
            local_max[1:-1] = (
                (local_score[1:-1] >= local_score[:-2])
                & (local_score[1:-1] > local_score[2:])
            )
        candidate_local_idx = np.flatnonzero(local_max & (local_score > 0))
        order = candidate_local_idx[np.argsort(local_score[candidate_local_idx])[::-1]]
        chosen = []
        for loc in order:
            glob = valid_idx[loc]
            if any(abs(glob - prev) < min_separation_bins for prev in chosen):
                continue
            chosen.append(glob)
            regions.append(np.array([loc]))
            if len(regions) >= n_features:
                break

    features = [build_feature(region) for region in regions]
    features = sorted(features, key=lambda feat: feat["region_score"], reverse=True)[:n_features]

    return {
        "continuum": continuum,
        "score": score,
        "excess_ratio": excess_ratio,
        "features": features,
    }

def plot_lst_fourier_spectrum(combined_uvp_dict,
                              combined_uvp_avg_dict,
                              pol="xx",
                              group_key=None,
                              dominant_n=5,
                              normalize="fractional",
                              window="hann",
                              fourier_method="auto",
                              remove_mean=True,
                              collapse_repeats=True,
                              bin_statistic="median",
                              duplicate_tol=None,
                              uniform_rtol=1e-5,
                              uniform_atol=1e-8,
                              plot_max_frequency=6.0,
                              feature_score_threshold=2.5,
                              continuum_window=19,
                              continuum_percentile=45,
                              max_cols=3):
    """
    Plot LST-axis Fourier amplitude spectra and highlight continuum-excess features.

    Fourier controls are passed to lst_fft_spectrum(). Use fourier_method='grid'
    for the original gridded diagnostic FFT, fourier_method='auto' to skip gridding
    when samples are already uniform and otherwise use a direct nonuniform Fourier
    sum, or fourier_method='direct' to always avoid gridding.
    """
    if group_key is None:
        group_key = list(combined_uvp_dict.keys())[-1]

    uvp = combined_uvp_dict[group_key]
    uvp_avg = combined_uvp_avg_dict[group_key]
    spw_items = sorted(get_spw_info(uvp).items(), key=lambda item: item[0])

    n_spws = len(spw_items)
    ncols = min(max_cols, max(1, n_spws))
    nrows = int(np.ceil(n_spws / ncols))
    subplot_width = 7.2
    subplot_height = 3.8
    fig_width = subplot_width * ncols
    fig_height = subplot_height * nrows + 0.45

    rc = {
        "figure.figsize": (fig_width, fig_height),
        "figure.dpi": 110,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7,
        "figure.titlesize": 11,
    }

    mode_summary = {}
    fft_info_summary = {}
    bl_vec = uvp.bl_vecs[0]
    baseline_length = np.linalg.norm(bl_vec)

    with plt.rc_context(rc):
        fig, axes = plt.subplots(nrows, ncols, squeeze=False, constrained_layout=True)
        axes = axes.ravel()

        for ax_idx, (spw_key, spw_val) in enumerate(spw_items):
            ax = axes[ax_idx]
            if spw_val is None:
                freq_range_str = "N/A"
            else:
                freq_min, freq_max = spw_val
                freq_range_str = f"{freq_min / 1e6:.2f}-{freq_max / 1e6:.2f} MHz"

            try:
                (_, _, _, _, lst_array_roll,
                 uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw_key)

                spectra = [
                    ("uvpspec_averaged_power", uvpspec_averaged_power, "royalblue", 1.0),
                    ("uvp_power", uvp_power, "crimson", 0.82),
                ]
                mode_summary[spw_key] = {}
                fft_info_by_label = {}

                for label, values, color, alpha in spectra:
                    freq, amp, _, _, fft_info = lst_fft_spectrum(
                        lst_array_roll,
                        values,
                        normalize=normalize,
                        window=window,
                        method=fourier_method,
                        statistic=bin_statistic,
                        collapse_repeats=collapse_repeats,
                        duplicate_tol=duplicate_tol,
                        remove_mean=remove_mean,
                        uniform_rtol=uniform_rtol,
                        uniform_atol=uniform_atol,
                        return_info=True,
                    )
                    fft_info_by_label[label] = fft_info
                    use = freq > 0
                    if plot_max_frequency is not None:
                        use &= freq <= plot_max_frequency

                    feature_result = spectral_information_filter(
                        freq,
                        amp,
                        n_features=dominant_n,
                        continuum_window=continuum_window,
                        continuum_percentile=continuum_percentile,
                        score_threshold=feature_score_threshold,
                        max_frequency=plot_max_frequency,
                    )
                    mode_summary[spw_key][label] = feature_result["features"]

                    ax.plot(freq[use], amp[use], color=color, alpha=alpha, linewidth=1.15, label=label)
                    continuum = feature_result["continuum"]
                    ax.plot(freq[use], continuum[use], color=color, linestyle="--", alpha=0.45, linewidth=0.9)

                    for feat in feature_result["features"]:
                        ax.axvspan(
                            feat["frequency_min_cyc_per_hr"],
                            feat["frequency_max_cyc_per_hr"],
                            color=color,
                            alpha=0.08,
                            linewidth=0,
                            zorder=0,
                        )

                    feature_freq = [feat["frequency_cyc_per_hr"] for feat in feature_result["features"]]
                    feature_amp = [feat["amplitude"] for feat in feature_result["features"]]
                    if feature_freq:
                        ax.scatter(
                            feature_freq,
                            feature_amp,
                            s=34,
                            marker="o",
                            facecolors="none",
                            edgecolors=color,
                            linewidths=1.4,
                            zorder=5,
                        )

                fft_info_summary[spw_key] = dict(fft_info_by_label)

            except Exception as exc:
                ax.text(0.5, 0.5, f"SPW {spw_key}\n{exc}", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue

            ax.set_yscale("log")
            # ax.set_xscale("log")
            ax.grid(alpha=0.22, linewidth=0.6)
            if plot_max_frequency is not None:
                ax.set_xlim(0, plot_max_frequency)
            ax.margins(x=0.02)
            ax.set_title(f"{baseline_length:.1f} m | SPW {spw_key} | {freq_range_str}", pad=4)
            if ax_idx // ncols == nrows - 1:
                ax.set_xlabel("LST Fourier frequency [cycles / hr]")
            if ax_idx % ncols == 0:
                ylabel = "FFT amplitude"
                if normalize == "fractional":
                    ylabel += "\n(fractional PSPEC)"
                ax.set_ylabel(ylabel)
            ax.legend(loc="best", frameon=True, borderpad=0.35, handlelength=1.8)

        for ax in axes[n_spws:]:
            fig.delaxes(ax)

        group_label = str(group_key)
        if len(group_label) > 52:
            group_label = group_label[:49] + "..."
        fig.suptitle(f"LST Fourier feature spectrum | group={group_label} | pol={pol}")
        plt.show()

    print("Feature-selected non-DC LST Fourier modes")
    print("score = robust log-amplitude excess above a smooth local continuum")
    for spw_key, label_modes in mode_summary.items():
        print(f"SPW {spw_key}:")
        for label, features in label_modes.items():
            if not features:
                print(f"  {label}: no feature above the current threshold")
                continue
            formatted = ", ".join(
                f"{feat['frequency_cyc_per_hr']:.4g} cyc/hr "
                f"[{feat['frequency_min_cyc_per_hr']:.4g}, {feat['frequency_max_cyc_per_hr']:.4g}] "
                f"(period {feat['period_hr']:.3g} hr, "
                f"excess x{feat['excess_ratio']:.2g}, "
                f"score {feat['score']:.2g}, region {feat['region_score']:.2g})"
                for feat in features
            )
            print(f"  {label}: {formatted}")
        for base_label, info in fft_info_summary.get(spw_key, {}).items():
            print(
                f"  {base_label} FFT: method={info['method']}, "
                f"N={info['n_samples']}, dt={info['dt_hours']:.5g} hr, "
                f"normalize={info['normalize']}, remove_mean={info['remove_mean']}, window={info['window']}"
            )

    return mode_summary


lst_fft_mode_summary = plot_lst_fourier_spectrum(
    combined_uvp_dict,
    combined_uvp_avg_dict,
    pol="xx",
    group_key=None,
    dominant_n=5,
    normalize="fractional",
    window="blackman-harris",
    fourier_method="direct",   # 'auto', 'grid', 'fft', or 'direct'/'nonuniform'
    remove_mean=True,         # set False to keep the DC mode in the transformed data
    collapse_repeats=True,    # collapse repeated baseline-pair samples at the same LST
    bin_statistic="median",   # repeated-LST/bin collapse statistic: 'median' or 'mean'
    plot_max_frequency=6.0,
    feature_score_threshold=2.5,
    continuum_window=15,
    continuum_percentile=40,
)


## Fringe spacing and FWHM from the RIME phase factor

Start with the geometric phase term in the scalar RIME for a baseline vector $\mathbf{b}$ at observing frequency $\nu$:

$$
V_{\mathbf{b}}(\nu)
= \int A(\hat{s},\nu) I(\hat{s},\nu)
\exp\left[-2\pi i\,\frac{\nu}{c}\,\mathbf{b}\cdot(\hat{s}-\hat{s}_0)\right] d\Omega .
$$

The exponential comes from the geometric delay between antennas. A plane wave from direction $\hat{s}$ reaches two antennas separated by $\mathbf{b}$ with delay

$$
\tau_g = \frac{\mathbf{b}\cdot(\hat{s}-\hat{s}_0)}{c},
$$

so the correlator sees the phase

$$
\phi = 2\pi \nu \tau_g
= 2\pi\frac{\mathbf{b}\cdot(\hat{s}-\hat{s}_0)}{\lambda},
\qquad
\lambda = \frac{c}{\nu}.
$$

Thus the complex fringe is

$$
\exp[-i\phi] = \cos\phi - i\sin\phi .
$$

The real cosine fringe is therefore not a separate physical assumption; it is the real part of the full complex phasor.

Now take an angular offset $\theta$ along the projected baseline direction. Then

$$
\mathbf{b}\cdot(\hat{s}-\hat{s}_0) \simeq b\sin\theta,
$$

and

$$
\phi(\theta) = 2\pi\frac{b}{\lambda}\sin\theta .
$$

### 1. Peak-to-peak fringe spacing

For the real fringe

$$
R(\theta)=\cos\phi(\theta),
$$

neighboring maxima occur when the phase changes by $2\pi$:

$$
\Delta\phi = 2\pi.
$$

Starting at the central maximum, $\theta=0$, the next maximum satisfies

$$
2\pi\frac{b}{\lambda}\sin\theta_{\rm fringe}=2\pi,
$$

so

$$
\boxed{\theta_{\rm fringe} = \arcsin\left(\frac{\lambda}{b}\right)}
$$

when $\lambda/b \le 1$. In the small-angle limit,

$$
\boxed{\theta_{\rm fringe} \simeq \frac{\lambda}{b}}
\qquad \mathrm{rad}.
$$

This is the usual interferometric fringe spacing or angular period near the phase center.

### 2. FWHM of the real amplitude fringe

For the central positive lobe of $R(\theta)=\cos\phi(\theta)$, the half-amplitude point satisfies

$$
\cos\phi_{1/2}=\frac{1}{2},
\qquad
\phi_{1/2}=\frac{\pi}{3}.
$$

Therefore

$$
2\pi\frac{b}{\lambda}\sin\theta_{1/2}=\frac{\pi}{3},
$$

which gives

$$
\sin\theta_{1/2}=\frac{\lambda}{6b}.
$$

The full width at half maximum is twice this one-sided angle:

$$
\boxed{\mathrm{FWHM}_{\rm amp}
=2\arcsin\left(\frac{\lambda}{6b}\right)}.
$$

For small angles,

$$
\boxed{\mathrm{FWHM}_{\rm amp} \simeq \frac{\lambda}{3b}}
\qquad \mathrm{rad}.
$$

### 3. FWHM of the power fringe

If the relevant quantity is power-like,

$$
P(\theta)=\cos^2\phi(\theta),
$$

then the half-power point satisfies

$$
\cos^2\phi_{1/2}=\frac{1}{2},
\qquad
\phi_{1/2}=\frac{\pi}{4}.
$$

Thus

$$
2\pi\frac{b}{\lambda}\sin\theta_{1/2}=\frac{\pi}{4},
$$

so

$$
\boxed{\mathrm{FWHM}_{\rm power}
=2\arcsin\left(\frac{\lambda}{8b}\right)}.
$$

For small angles,

$$
\boxed{\mathrm{FWHM}_{\rm power} \simeq \frac{\lambda}{4b}}
\qquad \mathrm{rad}.
$$

### 4. LST-hour conversion

Use

$$
24\ \mathrm{hr}_{\rm LST}=360^\circ=2\pi\ \mathrm{rad}.
$$

Therefore any angular width can be converted to LST hours by

$$
\boxed{\Delta t_{\rm LST}\,[\mathrm{hr}]
= \frac{\Delta\theta\,[\mathrm{deg}]}{15}
= \frac{12}{\pi}\,\Delta\theta\,[\mathrm{rad}]}.
$$

So the three useful widths are

$$
\boxed{\Delta t_{{\rm fringe},\,\rm LST}
= \frac{12}{\pi}\arcsin\left(\frac{\lambda}{b}\right)},
$$

$$
\boxed{\Delta t_{{\rm amp},\,\rm LST}
= \frac{24}{\pi}\arcsin\left(\frac{\lambda}{6b}\right)},
$$

and

$$
\boxed{\Delta t_{{\rm power},\,\rm LST}
= \frac{24}{\pi}\arcsin\left(\frac{\lambda}{8b}\right)}.
$$

Here $b$ is the projected baseline length in meters. A baseline length alone is not enough: the angular scale is set by $\lambda/b=c/(\nu b)$.


In [ ]:
# Fringe spacing, FWHM, LST conversion, and diagnostic plots

import numpy as np
import matplotlib.pyplot as plt

C_M_PER_S = 299_792_458.0
DEG_PER_LST_HOUR = 360.0 / 24.0
HERA_LATITUDE_DEG_FALLBACK = -30.7215271


def _default_hera_latitude_deg():
    """Return HERA latitude in degrees, preferring the packaged hera_sim value."""
    try:
        from hera_sim.io import HERA_LAT_LON_ALT
        return float(np.asarray(HERA_LAT_LON_ALT, dtype=float).ravel()[0]), "hera_sim.io.HERA_LAT_LON_ALT"
    except Exception:
        return HERA_LATITUDE_DEG_FALLBACK, "HERA_LATITUDE_DEG_FALLBACK"


def _latitude_deg_from_location_triplet(values, source_label):
    """Infer latitude from either lat/lon/alt degrees or ECEF xyz meters."""
    arr = np.asarray(values, dtype=float).ravel()
    if arr.size < 3 or not np.all(np.isfinite(arr[:3])):
        raise ValueError(f"Could not parse finite 3-vector from {source_label}.")

    x0, x1, x2 = arr[:3]
    if abs(x0) <= 90.0 and abs(x1) <= 360.0 and abs(x2) <= 1.0e5:
        return float(x0), f"{source_label} interpreted as lat/lon/alt deg"

    try:
        from astropy.coordinates import EarthLocation
        import astropy.units as u
        loc = EarthLocation.from_geocentric(x0 * u.m, x1 * u.m, x2 * u.m)
        return float(loc.lat.deg), f"{source_label} interpreted as ECEF xyz m"
    except Exception:
        radius_xy = np.hypot(x0, x1)
        if radius_xy <= 0:
            raise
        # Last-resort geocentric latitude. This is close, but not a full WGS84 geodetic conversion.
        return float(np.degrees(np.arctan2(x2, radius_xy))), f"{source_label} interpreted as geocentric xyz m"


def infer_instrument_latitude_deg(instrument=None):
    """Infer observatory latitude from a UVPSpec/UVData-like object, then fall back to HERA."""
    if instrument is not None:
        for attr in ("telescope_location_lat_lon_alt_degrees", "telescope_location_lat_lon_alt", "telescope_location"):
            if not hasattr(instrument, attr):
                continue
            values = getattr(instrument, attr)
            if values is None:
                continue
            try:
                if attr == "telescope_location_lat_lon_alt":
                    arr = np.asarray(values, dtype=float).ravel()
                    if arr.size >= 2 and abs(arr[0]) <= np.pi and abs(arr[1]) <= 2.0 * np.pi:
                        return float(np.degrees(arr[0])), f"instrument.{attr} interpreted as rad"
                return _latitude_deg_from_location_triplet(values, f"instrument.{attr}")
            except Exception:
                pass

    return _default_hera_latitude_deg()


def latitude_projection(latitude_deg=None, latitude_rad=None, instrument=None,
                        width_rad=None, width_deg=None, width_lst_hr=None):
    """
    Latitude projection for a constant-physical-width arc on a latitude circle.

    The circle at observatory latitude phi has circumference smaller than the
    equator by cos(phi). A fixed fringe spacing therefore maps to an effective
    24-hour sweep width larger by 1 / |cos(phi)|. Optional width inputs are
    returned after applying this scale.
    """
    if latitude_rad is not None:
        phi_rad = float(latitude_rad)
        latitude_deg = float(np.degrees(phi_rad))
        source = "latitude_rad input"
    elif latitude_deg is not None:
        latitude_deg = float(latitude_deg)
        phi_rad = float(np.radians(latitude_deg))
        source = "latitude_deg input"
    else:
        latitude_deg, source = infer_instrument_latitude_deg(instrument)
        phi_rad = float(np.radians(latitude_deg))

    cos_phi = abs(float(np.cos(phi_rad)))
    if not np.isfinite(cos_phi) or cos_phi <= 0:
        raise ValueError("Latitude projection is singular at the pole, where cos(phi)=0.")

    scale = 1.0 / cos_phi
    out = {
        "latitude_deg": latitude_deg,
        "latitude_rad": phi_rad,
        "cos_phi": cos_phi,
        "width_scale": scale,
        "source": source,
    }
    if width_rad is not None:
        out["width_phi_rad"] = np.asarray(width_rad, dtype=float) * scale
    if width_deg is not None:
        out["width_phi_deg"] = np.asarray(width_deg, dtype=float) * scale
    if width_lst_hr is not None:
        out["width_phi_lst_hr"] = np.asarray(width_lst_hr, dtype=float) * scale
    return out


def degrees_to_lst_hours(angle_deg):
    """Convert sky angle in degrees to equivalent LST hours using 24 hr = 360 deg."""
    return np.asarray(angle_deg, dtype=float) / DEG_PER_LST_HOUR


def radians_to_lst_hours(angle_rad):
    """Convert sky angle in radians to equivalent LST hours using 24 hr = 2 pi rad."""
    return np.asarray(angle_rad, dtype=float) * 12.0 / np.pi


def _wavelength_from_frequency_or_lambda(frequency_MHz=None, wavelength_m=None):
    if wavelength_m is not None:
        wavelength_m = float(wavelength_m)
        if wavelength_m <= 0:
            raise ValueError("wavelength_m must be positive.")
        frequency_MHz = C_M_PER_S / wavelength_m / 1e6
        return wavelength_m, frequency_MHz

    if frequency_MHz is None:
        raise ValueError("Provide frequency_MHz or wavelength_m; baseline length alone is insufficient.")

    frequency_MHz = float(frequency_MHz)
    if frequency_MHz <= 0:
        raise ValueError("frequency_MHz must be positive.")
    wavelength_m = C_M_PER_S / (frequency_MHz * 1e6)
    return wavelength_m, frequency_MHz


def _safe_arcsin_ratio(ratio):
    ratio = np.asarray(ratio, dtype=float)
    out = np.full_like(ratio, np.nan, dtype=float)
    ok = np.isfinite(ratio) & (np.abs(ratio) <= 1.0)
    out[ok] = np.arcsin(ratio[ok])
    if out.ndim == 0:
        return float(out)
    return out


def fringe_phase(theta_rad, b_m, frequency_MHz=None, wavelength_m=None):
    """RIME geometric fringe phase phi(theta) = 2 pi b sin(theta) / lambda."""
    wavelength_m, _ = _wavelength_from_frequency_or_lambda(
        frequency_MHz=frequency_MHz,
        wavelength_m=wavelength_m,
    )
    return 2.0 * np.pi * float(b_m) * np.sin(theta_rad) / wavelength_m


def fringe_widths(b_m, frequency_MHz=None, wavelength_m=None,
                  latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Return peak-to-peak fringe spacing and central-lobe FWHM values.

    Widths are returned in radians, degrees, and equivalent LST hours.
    The exact expressions keep sin(theta); the small-angle values use sin(theta) ~= theta.

    The additional fringe_phi_spacing entry applies the latitude-circle projection:

        width_phi = width / |cos(phi)|

    where phi is the observatory latitude inferred from instrument coordinates
    when available, or HERA's latitude otherwise.
    """
    b_m = float(b_m)
    if b_m <= 0:
        raise ValueError("b_m must be positive.")

    wavelength_m, frequency_MHz = _wavelength_from_frequency_or_lambda(
        frequency_MHz=frequency_MHz,
        wavelength_m=wavelength_m,
    )

    projection = latitude_projection(
        latitude_deg=latitude_deg,
        latitude_rad=latitude_rad,
        instrument=instrument,
    )
    phi_scale = projection["width_scale"]

    spacing_rad = _safe_arcsin_ratio(wavelength_m / b_m)
    spacing_small_rad = wavelength_m / b_m
    amp_fwhm_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (6.0 * b_m))
    power_fwhm_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (8.0 * b_m))
    null_to_null_rad = 2.0 * _safe_arcsin_ratio(wavelength_m / (4.0 * b_m))
    fringe_phi_spacing_rad = spacing_rad * phi_scale
    fringe_phi_spacing_small_rad = spacing_small_rad * phi_scale

    def bundle(angle_rad):
        return {
            "rad": angle_rad,
            "deg": np.degrees(angle_rad),
            "lst_hr": float(radians_to_lst_hours(angle_rad)),
        }

    return {
        "baseline_m": b_m,
        "frequency_MHz": frequency_MHz,
        "wavelength_m": wavelength_m,
        "lambda_over_b": wavelength_m / b_m,
        "latitude_projection": projection,
        "fringe_spacing": bundle(spacing_rad),
        "fringe_spacing_small_angle": bundle(spacing_small_rad),
        "fringe_phi_spacing": bundle(fringe_phi_spacing_rad),
        "fringe_phi_spacing_small_angle": bundle(fringe_phi_spacing_small_rad),
        "amp_fwhm": bundle(amp_fwhm_rad),
        "power_fwhm": bundle(power_fwhm_rad),
        "null_to_null": bundle(null_to_null_rad),
    }


def fringe_fwhm(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Backward-compatible FWHM helper returning radians, degrees, and LST hours."""
    widths = fringe_widths(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m)
    response = response.lower()
    if response in ("amplitude", "field", "cos"):
        values = widths["amp_fwhm"]
        response_name = "amplitude"
    elif response in ("power", "half-power", "cos2", "cos^2"):
        values = widths["power_fwhm"]
        response_name = "power"
    else:
        raise ValueError("response must be 'amplitude' or 'power'.")

    return {
        "response": response_name,
        "baseline_m": widths["baseline_m"],
        "frequency_MHz": widths["frequency_MHz"],
        "wavelength_m": widths["wavelength_m"],
        "fwhm_rad": values["rad"],
        "fwhm_deg": values["deg"],
        "fwhm_lst_hr": values["lst_hr"],
    }


def fringe_fwhm_deg(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Return only FWHM in degrees."""
    return fringe_fwhm(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m, response=response)["fwhm_deg"]


def fringe_fwhm_lst_hr(b_m, frequency_MHz=None, wavelength_m=None, response="amplitude"):
    """Return only FWHM in equivalent LST hours."""
    return fringe_fwhm(b_m, frequency_MHz=frequency_MHz, wavelength_m=wavelength_m, response=response)["fwhm_lst_hr"]


def print_fringe_width_table(b_m, frequency_MHz_list, latitude_deg=None, latitude_rad=None, instrument=None):
    """Print a compact table of fringe spacing, latitude-projected spacing, and FWHM quantities."""
    projection = latitude_projection(latitude_deg=latitude_deg, latitude_rad=latitude_rad, instrument=instrument)
    header = (
        "freq [MHz]  lambda [m]  spacing [deg/hr]  "
        "fringe(phi) [deg/hr]  amp FWHM [deg/hr]  power FWHM [deg/hr]"
    )
    print(
        f"Latitude projection: phi={projection['latitude_deg']:.4f} deg, "
        f"cos(phi)={projection['cos_phi']:.4f}, width scale=1/cos(phi)={projection['width_scale']:.4f} "
        f"[{projection['source']}]"
    )
    print(header)
    print("-" * len(header))
    for freq in frequency_MHz_list:
        w = fringe_widths(
            b_m,
            frequency_MHz=freq,
            latitude_deg=latitude_deg,
            latitude_rad=latitude_rad,
            instrument=instrument,
        )
        print(
            f"{freq:10.3f}  "
            f"{w['wavelength_m']:10.3f}  "
            f"{w['fringe_spacing']['deg']:7.3f}/{w['fringe_spacing']['lst_hr']:6.3f}  "
            f"{w['fringe_phi_spacing']['deg']:7.3f}/{w['fringe_phi_spacing']['lst_hr']:6.3f}  "
            f"{w['amp_fwhm']['deg']:7.3f}/{w['amp_fwhm']['lst_hr']:6.3f}  "
            f"{w['power_fwhm']['deg']:7.3f}/{w['power_fwhm']['lst_hr']:6.3f}"
        )


def plot_fringe_concepts_vs_phi():
    """Plot cos(phi) and cos^2(phi), showing phase-domain spacing and FWHM."""
    phi = np.linspace(-2.4 * np.pi, 2.4 * np.pi, 2000)
    amp = np.cos(phi)
    power = amp ** 2

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)

    ax = axes[0]
    ax.plot(phi / np.pi, amp, color="crimson", lw=1.8, label=r"$\cos\phi$")
    ax.axhline(0.5, color="black", ls="--", lw=1, alpha=0.7, label="half amplitude")
    ax.axvspan(-1/3, 1/3, color="crimson", alpha=0.12, label=r"amp FWHM: $2\pi/3$")
    for x in (-2, 0, 2):
        ax.axvline(x, color="0.25", lw=0.8, alpha=0.5)
    ax.annotate(
        "one fringe spacing",
        xy=(0, 1.05), xytext=(2, 1.05),
        arrowprops=dict(arrowstyle="<->", color="0.25"),
        ha="center", va="bottom", fontsize=9,
    )
    ax.set_xlabel(r"phase $\phi / \pi$")
    ax.set_ylabel("real fringe amplitude")
    ax.set_title(r"Amplitude fringe: $\cos\phi$")
    ax.set_ylim(-1.15, 1.2)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

    ax = axes[1]
    ax.plot(phi / np.pi, power, color="royalblue", lw=1.8, label=r"$\cos^2\phi$")
    ax.axhline(0.5, color="black", ls="--", lw=1, alpha=0.7, label="half power")
    ax.axvspan(-1/4, 1/4, color="royalblue", alpha=0.12, label=r"power FWHM: $\pi/2$")
    for x in (-2, 0, 2):
        ax.axvline(x, color="0.25", lw=0.8, alpha=0.5)
    ax.annotate(
        "one amplitude-fringe spacing",
        xy=(0, 1.05), xytext=(2, 1.05),
        arrowprops=dict(arrowstyle="<->", color="0.25"),
        ha="center", va="bottom", fontsize=9,
    )
    ax.set_xlabel(r"phase $\phi / \pi$")
    ax.set_ylabel("power fringe")
    ax.set_title(r"Power fringe: $\cos^2\phi$")
    ax.set_ylim(-0.05, 1.2)
    ax.grid(alpha=0.25)
    ax.legend(loc="lower right", fontsize=8)

    plt.show()


def plot_fringe_overlays_by_frequency(b_m, frequency_MHz_list, theta_limit_deg=None):
    """Plot real and power fringes versus angular offset for several frequencies."""
    frequency_MHz_list = np.asarray(frequency_MHz_list, dtype=float)
    if theta_limit_deg is None:
        widths = [fringe_widths(b_m, frequency_MHz=f)["fringe_spacing"]["deg"] for f in frequency_MHz_list]
        finite_widths = np.asarray(widths, dtype=float)[np.isfinite(widths)]
        theta_limit_deg = min(45.0, max(5.0, 1.6 * np.nanmax(finite_widths))) if finite_widths.size else 20.0

    theta_deg = np.linspace(-theta_limit_deg, theta_limit_deg, 2400)
    theta_rad = np.radians(theta_deg)

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(frequency_MHz_list)))

    for color, freq in zip(cmap, frequency_MHz_list):
        phi = fringe_phase(theta_rad, b_m, frequency_MHz=freq)
        wavelength_m = C_M_PER_S / (freq * 1e6)
        label = f"{freq:.1f} MHz, lambda={wavelength_m:.2f} m"
        axes[0].plot(theta_deg, np.cos(phi), color=color, lw=1.3, label=label)
        axes[1].plot(theta_deg, np.cos(phi) ** 2, color=color, lw=1.3, label=label)

        w = fringe_widths(b_m, frequency_MHz=freq)
        half_amp = 0.5 * w["amp_fwhm"]["deg"]
        half_power = 0.5 * w["power_fwhm"]["deg"]
        if np.isfinite(half_amp):
            axes[0].axvspan(-half_amp, half_amp, color=color, alpha=0.05, linewidth=0)
        if np.isfinite(half_power):
            axes[1].axvspan(-half_power, half_power, color=color, alpha=0.05, linewidth=0)

    axes[0].axhline(0.5, color="black", ls="--", lw=0.9, alpha=0.65)
    axes[1].axhline(0.5, color="black", ls="--", lw=0.9, alpha=0.65)
    axes[0].set_ylabel(r"$\cos\phi(\theta)$")
    axes[1].set_ylabel(r"$\cos^2\phi(\theta)$")
    axes[1].set_xlabel("angular offset along baseline [deg]")

    def deg_to_lst_hr(x):
        return np.asarray(x) / DEG_PER_LST_HOUR

    def lst_hr_to_deg(x):
        return np.asarray(x) * DEG_PER_LST_HOUR

    secax = axes[0].secondary_xaxis("top", functions=(deg_to_lst_hr, lst_hr_to_deg))
    secax.set_xlabel("equivalent LST offset [hr]")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"Fringe overlays for projected baseline b={b_m:.2f} m")
    plt.show()


def plot_fringe_widths_vs_frequency(b_m, frequency_MHz_grid, reference_frequency_MHz=None,
                                    latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Plot fringe spacing, amplitude FWHM, and power FWHM versus frequency.

    ROBUST_SINGLE_FREQUENCY_SWEEP: if the input grid is one frequency, or many
    copies of the same frequency, expand it to a local sweep so curves do not
    collapse to invisible zero-length line segments.
    """
    input_frequency_MHz = np.asarray(frequency_MHz_grid, dtype=float).ravel()
    input_frequency_MHz = input_frequency_MHz[np.isfinite(input_frequency_MHz) & (input_frequency_MHz > 0)]
    if input_frequency_MHz.size == 0:
        raise ValueError("frequency_MHz_grid must contain at least one positive finite frequency.")

    unique_frequency_MHz = np.unique(np.round(input_frequency_MHz, decimals=9))
    f_min = np.nanmin(input_frequency_MHz)
    f_max = np.nanmax(input_frequency_MHz)
    expanded_single_frequency = unique_frequency_MHz.size == 1 or np.isclose(f_min, f_max)

    if expanded_single_frequency:
        center = float(np.nanmedian(input_frequency_MHz))
        half_width = max(5.0, 0.10 * center)
        f_min = max(center - half_width, 1e-6)
        f_max = center + half_width
        plot_frequency_MHz = np.linspace(f_min, f_max, 240)
    else:
        plot_frequency_MHz = np.linspace(f_min, f_max, 240)

    spacing_deg = np.full(plot_frequency_MHz.size, np.nan)
    amp_deg = np.full(plot_frequency_MHz.size, np.nan)
    power_deg = np.full(plot_frequency_MHz.size, np.nan)
    phi_spacing_deg = np.full(plot_frequency_MHz.size, np.nan)

    for i, freq in enumerate(plot_frequency_MHz):
        w = fringe_widths(
            b_m,
            frequency_MHz=freq,
            latitude_deg=latitude_deg,
            latitude_rad=latitude_rad,
            instrument=instrument,
        )
        spacing_deg[i] = w["fringe_spacing"]["deg"]
        phi_spacing_deg[i] = w["fringe_phi_spacing"]["deg"]
        amp_deg[i] = w["amp_fwhm"]["deg"]
        power_deg[i] = w["power_fwhm"]["deg"]

    finite = (
        np.isfinite(plot_frequency_MHz)
        & np.isfinite(spacing_deg)
        & np.isfinite(phi_spacing_deg)
        & np.isfinite(amp_deg)
        & np.isfinite(power_deg)
    )
    print(
        "Width plot frequency range: "
        f"{plot_frequency_MHz[0]:.3f}-{plot_frequency_MHz[-1]:.3f} MHz; "
        f"finite points: {np.count_nonzero(finite)}/{plot_frequency_MHz.size}"
    )
    print(
        "Width plot y-ranges [deg]: "
        f"spacing {np.nanmin(spacing_deg):.3f}-{np.nanmax(spacing_deg):.3f}, "
        f"fringe(phi) {np.nanmin(phi_spacing_deg):.3f}-{np.nanmax(phi_spacing_deg):.3f}, "
        f"amp {np.nanmin(amp_deg):.3f}-{np.nanmax(amp_deg):.3f}, "
        f"power {np.nanmin(power_deg):.3f}-{np.nanmax(power_deg):.3f}"
    )
    if expanded_single_frequency:
        print("Input frequencies collapsed to one value; plotting a local +/-10% sweep around it.")

    fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
    marker_every = max(1, plot_frequency_MHz.size // 24)
    ax.plot(
        plot_frequency_MHz,
        spacing_deg,
        lw=2,
        marker="o",
        markevery=marker_every,
        ms=3,
        color="black",
        label=r"spacing $\approx \lambda/b$",
    )
    ax.plot(
        plot_frequency_MHz,
        phi_spacing_deg,
        lw=2,
        marker="D",
        markevery=marker_every,
        ms=3,
        color="darkorange",
        label=r"latitude-projected spacing $\lambda/[b\cos\phi]$",
    )
    ax.plot(
        plot_frequency_MHz,
        amp_deg,
        lw=2,
        marker="s",
        markevery=marker_every,
        ms=3,
        color="crimson",
        label="amplitude FWHM",
    )
    ax.plot(
        plot_frequency_MHz,
        power_deg,
        lw=2,
        marker="^",
        markevery=marker_every,
        ms=3,
        color="royalblue",
        label="power FWHM",
    )

    if reference_frequency_MHz is None:
        reference_frequency_MHz = unique_frequency_MHz
    reference_frequency_MHz = np.asarray(reference_frequency_MHz, dtype=float).ravel()
    reference_frequency_MHz = np.unique(reference_frequency_MHz[np.isfinite(reference_frequency_MHz) & (reference_frequency_MHz > 0)])
    for i, freq in enumerate(reference_frequency_MHz):
        label = "input/SPW frequency" if i == 0 else None
        ax.axvline(freq, color="0.25", ls=":", lw=1.1, alpha=0.65, label=label)

    ax.set_xlabel("frequency [MHz]")
    ax.set_ylabel("angular width [deg]")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")

    secax = ax.secondary_yaxis(
        "right",
        functions=(degrees_to_lst_hours, lambda h: np.asarray(h) * DEG_PER_LST_HOUR),
    )
    secax.set_ylabel("equivalent LST width [hr]")
    ax.set_title(f"Fringe widths versus frequency for projected baseline b={b_m:.2f} m")
    if expanded_single_frequency:
        ax.text(
            0.02,
            0.02,
            "single frequency input: showing local +/-10% sweep",
            transform=ax.transAxes,
            fontsize=8,
            color="0.25",
            ha="left",
            va="bottom",
        )
    plt.show()

# Example: infer a baseline and representative frequencies from the current UVPSpec dictionary.
try:
    example_uvp = next(iter(combined_uvp_dict.values()))
    example_instrument = example_uvp
    example_baseline_m = np.linalg.norm(example_uvp.bl_vecs[0])
    spw_centers_MHz = np.array([
        0.5 * (freq_range[0] + freq_range[1]) / 1e6
        for _, freq_range in sorted(get_spw_info(example_uvp).items(), key=lambda item: item[0])
        if freq_range is not None
    ])
except NameError:
    example_uvp = None
    example_instrument = None
    example_baseline_m = 29.216
    spw_centers_MHz = np.array([80.0, 120.0, 160.0, 200.0])

if spw_centers_MHz.size == 0:
    spw_centers_MHz = np.array([80.0, 120.0, 160.0, 200.0])

# Use a few representative frequencies for the overlay plot, and all available centers for the width trend.
overlay_indices = np.unique(np.linspace(0, spw_centers_MHz.size - 1, min(5, spw_centers_MHz.size)).round().astype(int))
overlay_frequency_MHz = spw_centers_MHz[overlay_indices]
frequency_grid_MHz = spw_centers_MHz

print(f"Example projected baseline: {example_baseline_m:.3f} m")
print_fringe_width_table(example_baseline_m, overlay_frequency_MHz, instrument=example_instrument)

plot_fringe_concepts_vs_phi()
plot_fringe_overlays_by_frequency(example_baseline_m, overlay_frequency_MHz)
plot_fringe_widths_vs_frequency(
    example_baseline_m,
    frequency_grid_MHz,
    reference_frequency_MHz=spw_centers_MHz,
    instrument=example_instrument,
)

# To use your own values:
# fringe_widths(14.6, frequency_MHz=150.0)  # uses HERA latitude fallback if no instrument is supplied
# fringe_fwhm_lst_hr(14.6, frequency_MHz=150.0, response="amplitude")
# plot_fringe_overlays_by_frequency(14.6, [100.0, 150.0, 200.0])


In [ ]:
# beam_lobe_widths -- ported from the (dropped) bandstop section; required by the
# Mode-Analytics analytic centers. Depends only on degrees_to_lst_hours and
# latitude_projection, both defined above.

def beam_lobe_widths(beam_main_lobe_width_deg, latitude_deg=None, latitude_rad=None, instrument=None):
    """
    Convert a beam main-lobe angular width into raw and latitude-projected LST spans.

    The raw mapping uses 24 LST hr = 360 deg, so width_hr = width_deg / 15.
    The projected mapping uses width_phi = width / |cos(phi)| for the latitude
    circle swept by the drift scan.
    """
    beam_main_lobe_width_deg = float(beam_main_lobe_width_deg)
    if not np.isfinite(beam_main_lobe_width_deg) or beam_main_lobe_width_deg <= 0:
        raise ValueError("beam_main_lobe_width_deg must be positive and finite.")

    raw_lst_hr = float(degrees_to_lst_hours(beam_main_lobe_width_deg))
    projection = latitude_projection(
        latitude_deg=latitude_deg,
        latitude_rad=latitude_rad,
        instrument=instrument,
        width_deg=beam_main_lobe_width_deg,
        width_lst_hr=raw_lst_hr,
    )
    phi_width_deg = float(np.asarray(projection["width_phi_deg"]))
    phi_width_lst_hr = float(np.asarray(projection["width_phi_lst_hr"]))

    return {
        "beam_lobe_width": {
            "deg": beam_main_lobe_width_deg,
            "lst_hr": raw_lst_hr,
        },
        "beam_lobe_phi_width": {
            "deg": phi_width_deg,
            "lst_hr": phi_width_lst_hr,
        },
        "latitude_projection": projection,
    }

In [ ]:
# Mode Analytics

import numpy as np
import matplotlib.pyplot as plt

# Pick one group/SPW/pol to view. None means use the same default selection as
# the bandstop cells: last group key and first valid SPW.
MODE_ANALYTICS_GROUP_KEY = globals().get("FRINGE_BANDSTOP_GROUP_KEY", globals().get("BANDSTOP_GROUP_KEY", None))
MODE_ANALYTICS_SPW = globals().get("FRINGE_BANDSTOP_SPW", globals().get("BANDSTOP_SPW", None))
MODE_ANALYTICS_POL = globals().get("FRINGE_BANDSTOP_POL", globals().get("BANDSTOP_POL", "xx"))

# Plot controls. Use MODE_ANALYTICS_XSCALE = "log" if desired.
MODE_ANALYTICS_MAX_FREQUENCY = 6.0
MODE_ANALYTICS_XSCALE = "log"
MODE_ANALYTICS_PLOT_MIXED_MODES = True
MODE_ANALYTICS_USE_PHI_MIXED_MODES = True  # True plots the cos(phi)-projected product modes.

MODE_ANALYTICS_WINDOW = "blackman-harris"  # "blackman-harris", "hann", None or "boxcar" gives no FFT window.
MODE_ANALYTICS_NORMALIZE = "none"   # "fractional" or None / "none" / "raw"
MODE_ANALYTICS_METHOD = "direct"            # "auto", "grid", "fft", "direct"/"nonuniform"
MODE_ANALYTICS_REMOVE_MEAN = True         # set False to keep DC
MODE_ANALYTICS_COLLAPSE_REPEATS = False    # collapse repeated LST samples
MODE_ANALYTICS_STATISTIC = "mean"       # "median" or "mean"
MODE_ANALYTICS_RETURN_INFO = False         # return additional information


# Analytic feature widths. These are converted to LST spans and then to
# Fourier centers by f_mode = 1 / Delta_LST.
MODE_ANALYTICS_FRINGE_SPACING_KEY = globals().get("FRINGE_MODE_SPACING_KEY", "fringe_phi_spacing")
MODE_ANALYTICS_LOBE_SPACING_KEY = globals().get("BEAM_LOBE_SPACING_KEY", "beam_lobe_phi_width")
MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG = 38.0
MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG = 18.0


def lst_span_to_fourier_mode(lst_span_hr):
    """Convert an LST feature width/span in hours into cycles per LST hour."""
    lst_span_hr = float(lst_span_hr)
    if not np.isfinite(lst_span_hr) or lst_span_hr <= 0:
        raise ValueError("lst_span_hr must be positive and finite.")
    return 1.0 / lst_span_hr


def fourier_mode_to_lst_span(mode_cyc_per_hr):
    """Convert a Fourier mode in cycles/hr back to its LST period/span."""
    mode_cyc_per_hr = float(mode_cyc_per_hr)
    if not np.isfinite(mode_cyc_per_hr) or mode_cyc_per_hr <= 0:
        return np.inf, np.inf
    span_lst_hr = 1.0 / mode_cyc_per_hr
    return span_lst_hr, span_lst_hr * DEG_PER_LST_HOUR


def select_mode_analytics_data(combined_uvp_dict, combined_uvp_avg_dict,
                               group_key=None, spw=None, pol="xx"):
    """Return one PSPEC-vs-LST track and its SPW metadata for the simple mode plot."""
    if group_key is None:
        group_key = list(combined_uvp_dict.keys())[-1]

    uvp = combined_uvp_dict[group_key]
    uvp_avg = combined_uvp_avg_dict[group_key]
    spw_info = get_spw_info(uvp)

    if spw is None:
        valid_spws = [key for key, val in sorted(spw_info.items()) if val is not None]
        if not valid_spws:
            raise ValueError("No valid SPW frequency ranges found.")
        spw = valid_spws[0]

    if spw_info.get(spw) is None:
        center_frequency_MHz = np.nan
        freq_range_str = "N/A"
    else:
        freq_min, freq_max = spw_info[spw]
        center_frequency_MHz = 0.5 * (freq_min + freq_max) / 1e6
        freq_range_str = f"{freq_min / 1e6:.2f}-{freq_max / 1e6:.2f} MHz"

    (_, _, _, _, lst_array_roll,
     uvp_power, uvpspec_averaged_power) = get_lst_pspec_for_fft(uvp, uvp_avg, pol, spw)

    return {
        "group_key": group_key,
        "spw": spw,
        "pol": pol,
        "uvp": uvp,
        "uvp_avg": uvp_avg,
        "baseline_m": float(np.linalg.norm(uvp.bl_vecs[0])),
        "center_frequency_MHz": center_frequency_MHz,
        "freq_range_str": freq_range_str,
        "lst_array_roll": lst_array_roll,
        "uvp_power": uvp_power,
        "uvpspec_averaged_power": uvpspec_averaged_power,
    }


def analytic_mode_centers(baseline_m, frequency_MHz, instrument=None,
                          fringe_spacing_key=MODE_ANALYTICS_FRINGE_SPACING_KEY,
                          lobe_spacing_key=MODE_ANALYTICS_LOBE_SPACING_KEY,
                          main_lobe_width_deg=MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG,
                          side_lobe_width_deg=MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG):
    """Compute the basic analytic centers for fringe, main-lobe, and side-lobe scales."""
    centers = []

    fringe = fringe_widths(baseline_m, frequency_MHz=frequency_MHz, instrument=instrument)
    fringe_span_hr = fringe[fringe_spacing_key]["lst_hr"]
    centers.append({
        "label": "fringe",
        "mode_cyc_per_hr": lst_span_to_fourier_mode(fringe_span_hr),
        "span_lst_hr": fringe_span_hr,
        "span_deg": fringe[fringe_spacing_key]["deg"],
        "color": "black",
        "linestyle": "-",
    })

    for label, width_deg, color, linestyle in [
        ("main lobe", main_lobe_width_deg, "darkorange", "--"),
        ("side lobe", side_lobe_width_deg, "seagreen", ":"),
    ]:
        lobe = beam_lobe_widths(width_deg, instrument=instrument)
        lobe_span_hr = lobe[lobe_spacing_key]["lst_hr"]
        centers.append({
            "label": label,
            "mode_cyc_per_hr": lst_span_to_fourier_mode(lobe_span_hr),
            "span_lst_hr": lobe_span_hr,
            "span_deg": lobe[lobe_spacing_key]["deg"],
            "input_width_deg": width_deg,
            "color": color,
            "linestyle": linestyle,
        })

    return centers


def product_mode_pair(label, mode_a, mode_b, projection, color):
    """
    Combine two Fourier modes from a product f_a(LST) f_b(LST).

    For two cosine-like features with modes k_a and k_b, multiplication gives
    product modes k_plus = k_a + k_b and k_minus = |k_a - k_b|.  The returned
    LST spans are lambda_plus = 1/k_plus and lambda_minus = 1/k_minus.
    """
    k_plus = mode_a + mode_b
    k_minus = abs(mode_a - mode_b)
    out = []
    for suffix, mode, linestyle in [("k+", k_plus, "-."), ("k-", k_minus, (0, (1, 1)))]:
        span_hr, span_deg = fourier_mode_to_lst_span(mode)
        out.append({
            "label": f"{label} {suffix}",
            "projection": projection,
            "mode_cyc_per_hr": mode,
            "span_lst_hr": span_hr,
            "span_deg": span_deg,
            "color": color,
            "linestyle": linestyle,
        })
    return out


def fringe_lobe_product_modes(baseline_m, frequency_MHz, lobe_width_deg, label,
                              instrument=None, color="purple"):
    """
    Return main/side-lobe x fringe product modes with and without cos(phi).

    The raw branch uses lambda/b fringe spacing and the input lobe width as-is.
    The phi branch uses the latitude-projected widths, width_phi = width/cos(phi).
    Both branches return k+ = k_lobe + k_fringe and k- = |k_lobe - k_fringe|,
    plus their equivalent LST wavelengths, lambda_LST = 1/k.
    """
    fringe = fringe_widths(baseline_m, frequency_MHz=frequency_MHz, instrument=instrument)
    lobe = beam_lobe_widths(lobe_width_deg, instrument=instrument)

    branches = {
        "raw": ("fringe_spacing", "beam_lobe_width", "no cos(phi)"),
        "phi": ("fringe_phi_spacing", "beam_lobe_phi_width", "cos(phi)"),
    }
    mixed = {}
    for key, (fringe_key, lobe_key, projection) in branches.items():
        k_fringe = lst_span_to_fourier_mode(fringe[fringe_key]["lst_hr"])
        k_lobe = lst_span_to_fourier_mode(lobe[lobe_key]["lst_hr"])
        modes = product_mode_pair(label, k_lobe, k_fringe, projection, color)
        for mode in modes:
            mode.update({
                "lobe_width_input_deg": lobe_width_deg,
                "lobe_mode_cyc_per_hr": k_lobe,
                "fringe_mode_cyc_per_hr": k_fringe,
                "lobe_span_lst_hr": lobe[lobe_key]["lst_hr"],
                "fringe_span_lst_hr": fringe[fringe_key]["lst_hr"],
                "lobe_span_deg": lobe[lobe_key]["deg"],
                "fringe_span_deg": fringe[fringe_key]["deg"],
            })
        mixed[key] = modes
    return mixed


def all_fringe_lobe_product_modes(baseline_m, frequency_MHz, instrument=None,
                                  main_lobe_width_deg=MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG,
                                  side_lobe_width_deg=MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG,
                                  use_phi=True):
    """Return all raw/phi product modes and the branch selected for plotting."""
    all_modes = {
        "main_lobe_x_fringe": fringe_lobe_product_modes(
            baseline_m, frequency_MHz, main_lobe_width_deg, "main x fringe", instrument=instrument, color="purple"
        ),
        "side_lobe_x_fringe": fringe_lobe_product_modes(
            baseline_m, frequency_MHz, side_lobe_width_deg, "side x fringe", instrument=instrument, color="teal"
        ),
    }
    branch = "phi" if use_phi else "raw"
    plotted = []
    for modes_by_projection in all_modes.values():
        plotted.extend(modes_by_projection[branch])
    return all_modes, plotted


def plot_simple_mode_analytics(combined_uvp_dict, combined_uvp_avg_dict,
                               group_key=MODE_ANALYTICS_GROUP_KEY,
                               spw=MODE_ANALYTICS_SPW,
                               pol=MODE_ANALYTICS_POL,
                               max_frequency=MODE_ANALYTICS_MAX_FREQUENCY,
                               window=MODE_ANALYTICS_WINDOW,
                               xscale=MODE_ANALYTICS_XSCALE,
                               plot_mixed_modes=MODE_ANALYTICS_PLOT_MIXED_MODES,
                               use_phi_mixed_modes=MODE_ANALYTICS_USE_PHI_MIXED_MODES,
                               normalize=MODE_ANALYTICS_NORMALIZE,
                               method=MODE_ANALYTICS_METHOD,
                               remove_mean=MODE_ANALYTICS_REMOVE_MEAN,
                               collapse_repeats=MODE_ANALYTICS_COLLAPSE_REPEATS,
                               statistic=MODE_ANALYTICS_STATISTIC,
                               return_info=MODE_ANALYTICS_RETURN_INFO
                               ):
    """Simple one-panel PSPEC FFT with analytic and product-mode centers as vertical lines."""
    data = select_mode_analytics_data(combined_uvp_dict, combined_uvp_avg_dict, group_key, spw, pol)
    print(method)
    freq_avg, amp_avg, _, _ = lst_fft_spectrum(
        data["lst_array_roll"],
        data["uvpspec_averaged_power"],
        normalize=normalize,
        window=window,
        method=method,
        remove_mean=remove_mean,
        collapse_repeats=collapse_repeats,
        statistic=statistic,
        return_info=return_info,
    )
    print(method)
    freq_uvp, amp_uvp, _, _ = lst_fft_spectrum(
        data["lst_array_roll"],
        data["uvp_power"],
        normalize=normalize,
        window=window,
        method=method,
        remove_mean=remove_mean,
        collapse_repeats=collapse_repeats,
        statistic=statistic,
        return_info=return_info,
    )

    use_avg = freq_avg > 0
    use_uvp = freq_uvp > 0
    if max_frequency is not None:
        use_avg &= freq_avg <= max_frequency
        use_uvp &= freq_uvp <= max_frequency

    centers = analytic_mode_centers(
        data["baseline_m"],
        data["center_frequency_MHz"],
        instrument=data["uvp"],
    )
    mixed_modes, plotted_mixed_centers = all_fringe_lobe_product_modes(
        data["baseline_m"],
        data["center_frequency_MHz"],
        instrument=data["uvp"],
        use_phi=use_phi_mixed_modes,
    )
    plot_centers = centers + (plotted_mixed_centers if plot_mixed_modes else [])

    fig, ax = plt.subplots(figsize=(9.5, 5.2), constrained_layout=True)
    ax.plot(freq_avg[use_avg], amp_avg[use_avg], color="royalblue", lw=1.2, label="uvpspec_averaged_power")
    ax.plot(freq_uvp[use_uvp], amp_uvp[use_uvp], color="crimson", lw=1.2, alpha=0.85, label="uvp_power")

    for center in plot_centers:
        f0 = center["mode_cyc_per_hr"]
        if not np.isfinite(f0) or f0 <= 0:
            continue
        ax.axvline(
            f0,
            color=center["color"],
            linestyle=center["linestyle"],
            lw=2.0,
            label=f"{center['label']}: {f0:.3g} cyc/hr",
        )

    ax.set_yscale("log")
    ax.set_xscale(xscale)

    positive_freq = np.concatenate([
        freq_avg[use_avg],
        freq_uvp[use_uvp],
        np.array([c["mode_cyc_per_hr"] for c in plot_centers], dtype=float),
    ])
    positive_freq = positive_freq[np.isfinite(positive_freq) & (positive_freq > 0)]
    if positive_freq.size and max_frequency is not None:
        ax.set_xlim(0.8 * np.nanmin(positive_freq), max_frequency)

    ax.grid(alpha=0.25, lw=0.6)
    ax.set_xlabel("LST Fourier frequency [cycles / hr]")
    ax.set_ylabel("FFT amplitude\n(fractional PSPEC)")
    ax.set_title(
        f"Analytic mode centers on PSPEC FFT | b={data['baseline_m']:.2f} m, "
        f"SPW {data['spw']} ({data['freq_range_str']}), pol={data['pol']}"
    )
    ax.legend(fontsize=8, loc="best")
    plt.show()

    print("Analytic centers:")
    for center in centers:
        print(
            f"  {center['label']}: {center['mode_cyc_per_hr']:.4g} cyc/hr; "
            f"span={center['span_lst_hr']:.4g} hr = {center['span_deg']:.4g} deg"
        )

    branch = "phi" if use_phi_mixed_modes else "raw"
    print(f"Product-mode centers plotted from branch: {branch}")
    for center in plotted_mixed_centers:
        print(
            f"  {center['label']}: {center['mode_cyc_per_hr']:.4g} cyc/hr; "
            f"lambda_LST={center['span_lst_hr']:.4g} hr = {center['span_deg']:.4g} deg; "
            f"from k_lobe={center['lobe_mode_cyc_per_hr']:.4g}, "
            f"k_fringe={center['fringe_mode_cyc_per_hr']:.4g} cyc/hr"
        )

    return {
        "fig": fig,
        "ax": ax,
        "data": data,
        "freq_avg": freq_avg,
        "amp_avg": amp_avg,
        "freq_uvp": freq_uvp,
        "amp_uvp": amp_uvp,
        "centers": centers,
        "mixed_modes": mixed_modes,
        "plotted_mixed_centers": plotted_mixed_centers,
    }


mode_analytics_result = plot_simple_mode_analytics(combined_uvp_dict, combined_uvp_avg_dict)


## Mode-Analytics across realizations

Individual seeds (`N_SAMPLE_REALIZATIONS`), then the **cosmic-variance mean of |FFT|** for both the coherent (`uvp_power`) and incoherent (`uvpspec_averaged_power`) tracks, overlaid with the analytic mode centers.

In [ ]:
# ===========================================================================
# Individual Mode-Analytics (LST-FFT + analytic centers) for a few realizations.
# Draws N_SAMPLE_REALIZATIONS evenly-spaced seeds using the existing
# plot_simple_mode_analytics() defined above.
# ===========================================================================
sample_idx = pick_sample_realizations(len(combined_uvp_dicts), N_SAMPLE_REALIZATIONS)
print(f"Individual Mode-Analytics for realizations: "
      f"{[REALIZATION_LABELS[r] for r in sample_idx]}")

per_realization_results = []
for r in sample_idx:
    print(f"\n---------------- realization {r}: {REALIZATION_LABELS[r]} ----------------")
    res = plot_simple_mode_analytics(combined_uvp_dicts[r], combined_uvp_avg_dicts[r])
    per_realization_results.append((r, REALIZATION_LABELS[r], res))

In [ ]:
# ===========================================================================
# COSMIC-VARIANCE AVERAGE of the Mode-Analytics LST-FFT spectra.
#
# For each realization we compute the |FFT| amplitude (the SAME quantity the
# individual plots show) for the coherent (uvp_power) and incoherent
# (uvpspec_averaged_power) tracks, then average |FFT| across realizations and
# overlay the analytic mode centers. Reuses lst_fft_spectrum,
# select_mode_analytics_data, analytic_mode_centers, all_fringe_lobe_product_modes.
# ===========================================================================
import numpy as np
import matplotlib.pyplot as plt

# Plot controls. Use MODE_ANALYTICS_XSCALE = "log" if desired.
MODE_ANALYTICS_MAX_FREQUENCY = 6.0
MODE_ANALYTICS_XSCALE = "log"
MODE_ANALYTICS_PLOT_MIXED_MODES = True
MODE_ANALYTICS_USE_PHI_MIXED_MODES = True  # True plots the cos(phi)-projected product modes.

MODE_ANALYTICS_WINDOW = "blackman-harris"  # "blackman-harris", "hann", None or "boxcar" gives no FFT window.
MODE_ANALYTICS_NORMALIZE = "none"   # "fractional" or None / "none" / "raw"
MODE_ANALYTICS_METHOD = "direct"            # "auto", "grid", "fft", "direct"/"nonuniform"
MODE_ANALYTICS_REMOVE_MEAN = True         # set False to keep DC
MODE_ANALYTICS_COLLAPSE_REPEATS = False    # collapse repeated LST samples
MODE_ANALYTICS_STATISTIC = "mean"       # "median" or "mean"
MODE_ANALYTICS_RETURN_INFO = False         # return additional information


# Analytic feature widths. These are converted to LST spans and then to
# Fourier centers by f_mode = 1 / Delta_LST.
MODE_ANALYTICS_FRINGE_SPACING_KEY = globals().get("FRINGE_MODE_SPACING_KEY", "fringe_phi_spacing")
MODE_ANALYTICS_LOBE_SPACING_KEY = globals().get("BEAM_LOBE_SPACING_KEY", "beam_lobe_phi_width")
MODE_ANALYTICS_MAIN_LOBE_WIDTH_DEG = 38.0
MODE_ANALYTICS_SIDE_LOBE_WIDTH_DEG = 18.0


def _align_to(freq_ref, freq, amp):
    """Return amp resampled onto freq_ref (identity if grids already match)."""
    amp = np.asarray(amp, float)
    if len(freq) == len(freq_ref) and np.allclose(freq, freq_ref):
        return amp
    return np.interp(freq_ref, np.asarray(freq, float), amp, left=np.nan, right=np.nan)


def _mode_fft_amplitudes(dd, da, group_key, spw, pol):
    """|FFT| amplitude for the coherent and incoherent LST tracks of one realization."""
    data = select_mode_analytics_data(dd, da, group_key, spw, pol)
    f_avg, a_avg, _, _ = lst_fft_spectrum(
        data["lst_array_roll"], data["uvpspec_averaged_power"],
        normalize=MODE_ANALYTICS_NORMALIZE, window=MODE_ANALYTICS_WINDOW,
        method=MODE_ANALYTICS_METHOD, remove_mean=MODE_ANALYTICS_REMOVE_MEAN,
        collapse_repeats=MODE_ANALYTICS_COLLAPSE_REPEATS,
        statistic=MODE_ANALYTICS_STATISTIC, return_info=False)
    f_uvp, a_uvp, _, _ = lst_fft_spectrum(
        data["lst_array_roll"], data["uvp_power"],
        normalize=MODE_ANALYTICS_NORMALIZE, window=MODE_ANALYTICS_WINDOW,
        method=MODE_ANALYTICS_METHOD, remove_mean=MODE_ANALYTICS_REMOVE_MEAN,
        collapse_repeats=MODE_ANALYTICS_COLLAPSE_REPEATS,
        statistic=MODE_ANALYTICS_STATISTIC, return_info=False)
    return data, f_avg, a_avg, f_uvp, a_uvp


def plot_averaged_mode_analytics(combined_uvp_dicts, combined_uvp_avg_dicts, labels,
                                 group_key=MODE_ANALYTICS_GROUP_KEY,
                                 spw=MODE_ANALYTICS_SPW,
                                 pol=MODE_ANALYTICS_POL,
                                 max_frequency=MODE_ANALYTICS_MAX_FREQUENCY,
                                 xscale=MODE_ANALYTICS_XSCALE,
                                 plot_mixed_modes=MODE_ANALYTICS_PLOT_MIXED_MODES,
                                 use_phi_mixed_modes=MODE_ANALYTICS_USE_PHI_MIXED_MODES,
                                 show_individual=True):
    """Mean |FFT| over realizations for coherent + incoherent, with analytic centers."""
    freq_ref, ref_data = None, None
    amps_avg, amps_uvp, used = [], [], []

    for r in range(len(combined_uvp_dicts)):
        if group_key is not None and group_key not in combined_uvp_dicts[r]:
            continue
        try:
            data, f_avg, a_avg, f_uvp, a_uvp = _mode_fft_amplitudes(
                combined_uvp_dicts[r], combined_uvp_avg_dicts[r], group_key, spw, pol)
        except Exception as exc:
            print(f"[skip r{r} {labels[r]}] {exc}")
            continue
        if freq_ref is None:
            freq_ref, ref_data = f_avg, data
        amps_avg.append(_align_to(freq_ref, f_avg, a_avg))
        amps_uvp.append(_align_to(freq_ref, f_uvp, a_uvp))
        used.append(labels[r])

    if not amps_avg:
        raise RuntimeError("No realizations produced a spectrum to average.")

    stack_avg, stack_uvp = np.vstack(amps_avg), np.vstack(amps_uvp)
    mean_avg, mean_uvp = np.nanmean(stack_avg, axis=0), np.nanmean(stack_uvp, axis=0)
    std_avg, std_uvp = np.nanstd(stack_avg, axis=0), np.nanstd(stack_uvp, axis=0)

    centers = analytic_mode_centers(ref_data["baseline_m"], ref_data["center_frequency_MHz"],
                                    instrument=ref_data["uvp"])
    _, plotted_mixed = all_fringe_lobe_product_modes(
        ref_data["baseline_m"], ref_data["center_frequency_MHz"],
        instrument=ref_data["uvp"], use_phi=use_phi_mixed_modes)
    plot_centers = centers + (plotted_mixed if plot_mixed_modes else [])

    use = freq_ref > 0
    if max_frequency is not None:
        use &= freq_ref <= max_frequency

    fig, ax = plt.subplots(figsize=(9.8, 5.4), constrained_layout=True)
    if show_individual:
        for a in amps_avg:
            ax.plot(freq_ref[use], a[use], color="royalblue", lw=0.5, alpha=0.22)
        for a in amps_uvp:
            ax.plot(freq_ref[use], a[use], color="crimson", lw=0.5, alpha=0.18)
    ax.plot(freq_ref[use], mean_avg[use], color="navy", lw=2.2,
            label=f"<|FFT| uvpspec_averaged_power>  ({len(amps_avg)} rlzns)")
    ax.plot(freq_ref[use], mean_uvp[use], color="darkred", lw=2.2,
            label=f"<|FFT| uvp_power>  ({len(amps_uvp)} rlzns)")

    for center in plot_centers:
        f0 = center["mode_cyc_per_hr"]
        if not np.isfinite(f0) or f0 <= 0:
            continue
        ax.axvline(f0, color=center["color"], linestyle=center["linestyle"], lw=2.0,
                   label=f"{center['label']}: {f0:.3g} cyc/hr")

    ax.set_yscale("log")
    ax.set_xscale(xscale)
    posf = freq_ref[use]
    posf = posf[np.isfinite(posf) & (posf > 0)]
    if posf.size and max_frequency is not None:
        ax.set_xlim(0.8 * np.nanmin(posf), max_frequency)
    ax.grid(alpha=0.25, lw=0.6)
    ax.set_xlabel("LST Fourier frequency [cycles / hr]")
    ax.set_ylabel("mean |FFT| amplitude\n(averaged over realizations)")
    ax.set_title(f"CV-averaged PSPEC FFT (mean |FFT| over {len(amps_avg)} realizations) | "
                 f"b={ref_data['baseline_m']:.2f} m, SPW {ref_data['spw']} "
                 f"({ref_data['freq_range_str']}), pol={pol}")
    ax.legend(fontsize=8, loc="best")
    plt.show()

    print(f"Averaged over realizations: {used}")
    print("Analytic centers:")
    for center in centers:
        print(f"  {center['label']}: {center['mode_cyc_per_hr']:.4g} cyc/hr; "
              f"span={center['span_lst_hr']:.4g} hr = {center['span_deg']:.4g} deg")

    return {"freq": freq_ref, "mean_avg": mean_avg, "mean_uvp": mean_uvp,
            "std_avg": std_avg, "std_uvp": std_uvp, "centers": centers,
            "plotted_mixed": plotted_mixed, "n_realizations": len(amps_avg),
            "labels_used": used}


averaged_mode_result = plot_averaged_mode_analytics(
    combined_uvp_dicts, combined_uvp_avg_dicts, REALIZATION_LABELS)